> ⚠️ **LEGACY — pending the unified GPU/CPU/FPGA runner refactor.**
>
> This notebook reads the `results/benchmark/<model>/benchmark_{gpu,cpu,fpga}_<scenario>.json` schema produced by the **removed** `run_full_benchmark.py` + `benchmark_fpga.py` (Python era). The current C++ FPGA benchmark (`benchmark_hardware`) uses a **different schema** (configs s0/s1, `stages`, ceilings) — see `docs/FPGA_benchmark.md`. Existing `results/benchmark/` JSONs still plot, but the FPGA half cannot be regenerated here. Cross-platform comparison will be rebuilt with the **unified runner** (`docs/FPGA_benchmark.md` §10, item [E]). For current FPGA-only analysis use `benchmark_hardware_analysis.ipynb`.

# Performance Benchmark Analysis — GPU / CPU / FPGA

Cross-platform comparison of **latency**, **throughput**, **power**, and **energy efficiency**
for the SAR DDC hyper-autoencoder across three backends:

| Backend | Hardware | Precision | Entropy Coding |
|---|---|---|---|
| **GPU** | NVIDIA RTX A4000 (16 GB) | FP32 | CompressAI Python (x86) |
| **CPU** | Host x86 (same machine) | FP32 | CompressAI Python (x86) |
| **FPGA** | Xilinx ZCU102 (DPU B4096) | INT8 | C++ rANS (`ans.so`, ARM A53) |

**Data sources:** JSON files produced by `benchmark_gpu.py` and `benchmark_fpga.py`
via the orchestrator `run_full_benchmark.py`.

**Structure:**
1. Setup & configuration
2. Load & merge all JSON results → unified DataFrame
3. Overview table
4. End-to-end latency comparison (bar chart)
5. Per-step latency breakdown (stacked bars)
6. NN vs entropy latency split
7. Throughput comparison
8. Power consumption
9. Energy per inference (the fairest cross-platform metric)
10. Summary table for publication

## 1 · Setup & Configuration

In [ ]:
import json
import warnings
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.axes import Axes
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)

# ── Paths ──────────────────────────────────────────────────────────────────────
ROOT_DIR = Path("..").resolve()

# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  SET THIS to the model you want to analyse                                   ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
BENCHMARK_DIR = ROOT_DIR / "results" / "benchmark" / "ResSHyp-relu_s1_L1000_pt"

assert BENCHMARK_DIR.exists(), f"Benchmark directory not found: {BENCHMARK_DIR}"
MODEL_NAME = BENCHMARK_DIR.name
print(f"Analysing: {MODEL_NAME}")
print(f"Directory: {BENCHMARK_DIR}")
print(f"JSON files found: {sorted(p.name for p in BENCHMARK_DIR.glob('*.json'))}")

# ── Load shared color palette (Okabe-Ito, colorblind-safe) ────────────────────
C = json.load(open(ROOT_DIR / "notebooks" / "plots_colors.json"))

# ── Plotting style ─────────────────────────────────────────────────────────────
PLATFORM_COLORS = {
    "gpu": C["platforms"]["gpu_dynamic"],
    "cpu": C["platforms"]["cpu_dynamic"],
    "fpga": C["platforms"]["fpga_dynamic"],
}
PLATFORM_LABELS = {"gpu": "GPU (RTX A4000)", "cpu": "CPU (Intel Xeon)", "fpga": "FPGA (ZCU102)"}
PLATFORM_ORDER = ["gpu", "cpu", "fpga"]

# Scenario display names & ordering
SCENARIO_ORDER = ["full", "compress", "decompress", "nn_only", "entropy_only"]
SCENARIO_LABELS = {
    "full": "Full",
    "compress": "Compress",
    "decompress": "Decompress",
    "nn_only": "NN Only",
    "entropy_only": "Entropy Only",
}


SAVE_FIGURES = True
FIGURE_FORMAT = "pdf"
PLOTS_DIR = ROOT_DIR / "results" / "plots" / "benchmark"

plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "font.size": 10,
    }
)


## 2 · Load & Merge All JSON Results

Each JSON file is one `(platform, scenario)` measurement. File naming conventions:

| Pattern | Platform | Source script |
|---|---|---|
| `benchmark_gpu_<scenario>.json` | GPU (CUDA) | `benchmark_gpu.py` |
| `benchmark_cpu_<scenario>.json` | CPU (x86 host) | `benchmark_gpu.py --no-gpu` |
| `benchmark_fpga_<scenario>.json` | FPGA (ZCU102 DPU) | `benchmark_fpga.py` |

All three share the same top-level schema (see `FPGA_benchmark.md`
§5.5 and §9.8).  Key differences:

- **Step label prefixes**: `gpu_` (GPU NN), `nn_` (CPU NN), `dpu_` (FPGA NN),
  `cpu_` (entropy / data ops on all platforms).
- **Power keys**: GPU → `power.gpu` (nvidia-smi) + `power.cpu_rapl`;
  FPGA → `power.per_rail` (INA226) + `power.groups_avg_w`.
- **NN-only scenario**: all platforms now use `nn_only` uniformly.
  different label).

In [ ]:
def _classify_file(name: str) -> Tuple[str, str]:
    """Return (platform, scenario) from a benchmark JSON filename.

    Naming conventions:
      benchmark_gpu_<scenario>.json   → ("gpu",  scenario)
      benchmark_cpu_<scenario>.json   → ("cpu",  scenario)
      benchmark_fpga_<scenario>.json  → ("fpga", scenario)

    Legacy: ``benchmark_<scenario>.json`` (no platform tag) is treated as FPGA.
    Sequential FPGA files (``benchmark_fpga_<scenario>_sequential.json``) are
    identified here but filtered out in ``load_all_results`` — they are used
    exclusively in §11 (parallel vs sequential comparison).
    """
    stem = name.replace(".json", "")
    parts = stem.split("_")  # e.g. ["benchmark", "gpu", "full"]

    if len(parts) >= 3 and parts[1] in ("gpu", "cpu", "fpga"):
        platform = parts[1]
        scenario = "_".join(parts[2:])
    else:
        # Legacy FPGA: benchmark_<scenario>.json (no platform prefix)
        platform = "fpga"
        scenario = "_".join(parts[1:])

    return platform, scenario


# ── FPGA power-rail group assignments ─────────────────────────────────────────
# Must mirror POWER_GROUPS in scripts/fpga/benchmark_fpga.py.
# See docs/FPGA_benchmark.md §4.7 for full definitions.
_FPGA_PL_RAILS = frozenset({"VCCINT", "VCCBRAM", "VCCAUX", "VCC1V2", "VCC3V3"})
_FPGA_PS_RAILS = frozenset(
    {
        "VCCPSINTFP",
        "VCCPSINTLP",
        "VCCPSAUX",
        "VCCPSPLL",
        "VCCO_PSDDR_504",
        "VCCOPS",
        "VCCOPS3",
        "VCCPSDDRPLL",
    }
)
_FPGA_MGT_RAILS = frozenset({"MGTAVCC", "MGTAVTT", "MGTRAVCC", "MGTRAVTT"})
_FPGA_FMC_RAILS = frozenset({"VADJ_FMC"})
_FPGA_DPU_RAILS = frozenset({"VCCINT", "VCCBRAM"})  # DPU_fabric
_FPGA_PS_COMP_RAILS = frozenset({"VCCPSINTFP", "VCCPSINTLP"})  # PS_compute (ARM A53 cores)


def _extract_power(data: dict, platform: str) -> Dict[str, Optional[float]]:
    """Extract power metrics into a flat dict, regardless of platform schema.

    Returned keys (all Optional[float]):
      power_total_w        — total load power (FPGA→TBP; GPU→GPU+RAPL; CPU→RAPL)
      power_nn_w           — NN accelerator power (DPU_fabric / GPU board)
      power_cpu_w          — CPU / entropy domain power (PS_compute / RAPL)
      power_idle_total_w   — idle baseline matching the scope of power_total_w
      energy_total_j       — total energy over the measurement window

    FPGA-specific group keys (load / idle):
      power_PL_w, power_PS_w, power_MGT_w, power_peripherals_w,
      power_TBP_w, power_MPSoC_w, power_DPU_fabric_w, power_PS_compute_w
      idle_PL_w,  idle_PS_w,  idle_MGT_w,  idle_peripherals_w,
      idle_TBP_w, idle_MPSoC_w, idle_DPU_fabric_w, idle_PS_compute_w

    GPU/CPU-specific:
      power_gpu_w, power_cpu_rapl_w, idle_gpu_w, idle_cpu_rapl_w
    """
    pw = data.get("power", {})
    out: Dict[str, Optional[float]] = {
        "power_total_w": None,
        "power_nn_w": None,
        "power_cpu_w": None,
        "power_idle_total_w": None,
        "energy_total_j": None,
        # FPGA groups (load)
        "power_PL_w": None,
        "power_PS_w": None,
        "power_MGT_w": None,
        "power_peripherals_w": None,
        "power_TBP_w": None,
        "power_MPSoC_w": None,
        "power_DPU_fabric_w": None,
        "power_PS_compute_w": None,
        # FPGA groups (idle)
        "idle_PL_w": None,
        "idle_PS_w": None,
        "idle_MGT_w": None,
        "idle_peripherals_w": None,
        "idle_TBP_w": None,
        "idle_MPSoC_w": None,
        "idle_DPU_fabric_w": None,
        "idle_PS_compute_w": None,
        # GPU / CPU
        "power_gpu_w": None,
        "power_cpu_rapl_w": None,
        "idle_gpu_w": None,
        "idle_cpu_rapl_w": None,
    }
    if not pw:
        return out

    if platform == "fpga":
        groups = pw.get("groups_avg_w", {})
        # TBP = MPSoC + MGT + peripherals (= board_total_extended_avg_w in JSON).
        # FMC (VADJ_FMC = 0.0 W on ZCU102, slot unused) is excluded from groups["TBP"]
        # in benchmark_fpga.py — add explicitly here for scope completeness.
        _tbp = groups.get("TBP") or pw.get("board_total_extended_avg_w")
        _fmc = groups.get("FMC", 0.0) or 0.0
        out["power_total_w"] = round(_tbp + _fmc, 4) if _tbp is not None else None
        out["power_TBP_w"] = out["power_total_w"]
        out["power_MPSoC_w"] = groups.get("MPSoC")
        out["power_PL_w"] = groups.get("PL")
        out["power_PS_w"] = groups.get("PS")
        out["power_MGT_w"] = groups.get("MGT")
        out["power_peripherals_w"] = groups.get("peripherals")
        out["power_DPU_fabric_w"] = groups.get("DPU_fabric")
        out["power_PS_compute_w"] = groups.get("PS_compute")
        # Canonical NN / CPU aliases used by generic plots
        out["power_nn_w"] = groups.get("DPU_fabric")
        out["power_cpu_w"] = groups.get("PS_compute")

        # Energy: TBP × measurement duration
        duration = next(
            (
                v["duration_s"]
                for v in pw.get("per_rail", {}).values()
                if isinstance(v, dict) and "duration_s" in v
            ),
            None,
        )
        if out["power_total_w"] is not None and duration:
            out["energy_total_j"] = out["power_total_w"] * duration

        # ── Idle baseline per power group ──────────────────────────────────────
        idle = pw.get("idle_baseline", {})

        def _rs(rails: frozenset) -> Optional[float]:
            if not idle:
                return None
            return round(sum(idle.get(r, {}).get("avg_power_w", 0.0) for r in rails), 4)

        out["idle_PL_w"] = _rs(_FPGA_PL_RAILS)
        out["idle_PS_w"] = _rs(_FPGA_PS_RAILS)
        out["idle_MGT_w"] = _rs(_FPGA_MGT_RAILS)
        out["idle_DPU_fabric_w"] = _rs(_FPGA_DPU_RAILS)
        out["idle_PS_compute_w"] = _rs(_FPGA_PS_COMP_RAILS)

        idle_ina226 = pw.get("idle_board_total_avg_w") or (
            sum(v.get("avg_power_w", 0.0) for v in idle.values() if isinstance(v, dict))
            if idle
            else 0.0
        )
        idle_pmbus = pw.get("idle_pmbus_rails", {})
        idle_periph = round(
            sum(v.get("avg_power_w", 0.0) for v in idle_pmbus.values() if isinstance(v, dict)), 4
        )
        out["idle_peripherals_w"] = idle_periph if idle_pmbus else None
        if idle:
            out["power_idle_total_w"] = round(idle_ina226 + idle_periph, 4)
            out["idle_TBP_w"] = out["power_idle_total_w"]
        if out["idle_PL_w"] is not None and out["idle_PS_w"] is not None:
            out["idle_MPSoC_w"] = round(out["idle_PL_w"] + out["idle_PS_w"], 4)

    elif platform == "gpu":
        gpu_pw = pw.get("gpu", {})
        rapl_total = pw.get("cpu_rapl_total_avg_w") or 0.0
        gpu_w = gpu_pw.get("avg_power_w", 0.0) if gpu_pw else 0.0

        out["power_gpu_w"] = gpu_w if gpu_pw else None
        out["power_cpu_rapl_w"] = rapl_total if rapl_total else None
        out["power_nn_w"] = gpu_w
        out["power_cpu_w"] = rapl_total if rapl_total else None
        out["power_total_w"] = gpu_w + (rapl_total or 0.0)

        gpu_e = gpu_pw.get("energy_j", 0.0) if gpu_pw else 0.0
        rapl_e = sum(
            v.get("energy_j", 0.0) for v in pw.get("cpu_rapl", {}).values() if isinstance(v, dict)
        )
        out["energy_total_j"] = gpu_e + rapl_e

        idle_gpu = pw.get("idle_gpu", {})
        idle_rapl = pw.get("idle_cpu_rapl", {})
        idle_g = idle_gpu.get("avg_power_w", 0.0) if idle_gpu else 0.0
        idle_r = sum(v.get("avg_power_w", 0.0) for v in idle_rapl.values() if isinstance(v, dict))
        out["idle_gpu_w"] = idle_g if idle_gpu else None
        out["idle_cpu_rapl_w"] = idle_r if idle_rapl else None
        if idle_gpu or idle_rapl:
            out["power_idle_total_w"] = idle_g + idle_r

    elif platform == "cpu":
        rapl_total = pw.get("cpu_rapl_total_avg_w") or 0.0
        out["power_nn_w"] = None
        out["power_cpu_w"] = rapl_total if rapl_total else None
        out["power_total_w"] = rapl_total if rapl_total else None
        out["power_cpu_rapl_w"] = rapl_total if rapl_total else None

        rapl_e = sum(
            v.get("energy_j", 0.0) for v in pw.get("cpu_rapl", {}).values() if isinstance(v, dict)
        )
        out["energy_total_j"] = rapl_e if rapl_e else None

        idle_rapl = pw.get("idle_cpu_rapl", {})
        if idle_rapl:
            idle_r = sum(
                v.get("avg_power_w", 0.0) for v in idle_rapl.values() if isinstance(v, dict)
            )
            out["idle_cpu_rapl_w"] = idle_r
            out["power_idle_total_w"] = idle_r

    return out


def _extract_step_breakdown(data: dict, platform: str) -> Dict[str, float]:
    """Return {step_label: mean_ms} from the latency_breakdown, renaming prefixes
    to canonical 'nn_' and 'cpu_' for uniform cross-platform comparison.

    Original prefixes: gpu_ (GPU), nn_ (CPU-as-NN), dpu_ (FPGA).
    Canonical:         nn_ (NN accelerator), cpu_ (entropy/data ops).
    """
    breakdown = data.get("latency_breakdown", {})
    out: Dict[str, float] = {}
    for step, stats in breakdown.items():
        if not isinstance(stats, dict):
            continue
        mean_s = stats.get("mean_s", 0.0)
        # Rename platform-specific NN prefixes to canonical "nn_"
        canonical = step
        if platform == "gpu" and step.startswith("gpu_"):
            canonical = "nn_" + step[4:]
        elif platform == "fpga" and step.startswith("dpu_"):
            canonical = "nn_" + step[4:]
        out[canonical] = mean_s * 1000  # convert to ms
    return out


def load_all_results(benchmark_dir: Path) -> pd.DataFrame:
    """Load all benchmark_*.json files from a directory into a tidy DataFrame.

    One row per (platform, scenario).  Columns include latency, throughput,
    power, and per-step breakdown (as a nested dict for later expansion).

    Sequential FPGA files (``benchmark_fpga_<scenario>_sequential.json``) are
    excluded here and loaded directly in §11 (parallel vs sequential comparison).
    They are NOT included in the main ``df`` DataFrame used by §2–§10.
    """
    rows: List[Dict[str, Any]] = []
    skipped_seq: List[str] = []

    for jpath in sorted(benchmark_dir.glob("benchmark_*.json")):
        platform, scenario = _classify_file(jpath.name)
        # Sequential FPGA files are not part of the main analysis.
        # They are loaded directly from disk in §11.
        if scenario.endswith("_sequential"):
            skipped_seq.append(jpath.name)
            continue
        with open(jpath) as f:
            data: dict = json.load(f)

        # ── Core metrics ───────────────────────────────────────────────
        # Latency (ms)
        latency_ms = data.get("latency_total_mean_ms", 0.0)

        # NN vs CPU latency split — keys differ by platform
        nn_key = next(
            (
                k
                for k in data
                if k.startswith("latency_")
                and k.endswith("_total_mean_ms")
                and k not in ("latency_total_mean_ms", "latency_cpu_total_mean_ms")
            ),
            None,
        )
        nn_latency_ms = data.get(nn_key, 0.0) if nn_key else 0.0
        cpu_latency_ms = data.get("latency_cpu_total_mean_ms", 0.0)

        # Throughput
        throughput_fps = data.get("throughput_fps", 0.0)

        # Compressed bytes → BPP
        avg_bytes = data.get("avg_compressed_bytes")
        bpp = (avg_bytes * 8 / (256 * 256)) if avg_bytes else None

        # Power
        power = _extract_power(data, platform)

        # Per-step breakdown (kept as dict — expanded in dedicated section)
        steps = _extract_step_breakdown(data, platform)

        # Energy per inference:  E_tile = P_total x t_tile
        energy_per_tile_mj = None
        if power["power_total_w"] is not None and latency_ms > 0:
            energy_per_tile_mj = power["power_total_w"] * (latency_ms / 1000) * 1000  # mJ

        row: Dict[str, Any] = {
            "platform": platform,
            "scenario": scenario,
            "source_file": jpath.name,
            # Latency
            "latency_ms": latency_ms,
            "nn_latency_ms": nn_latency_ms,
            "cpu_latency_ms": cpu_latency_ms,
            "nn_fraction": nn_latency_ms / latency_ms if latency_ms > 0 else 0,
            # Throughput
            "throughput_fps": throughput_fps,
            # Compression
            "avg_compressed_bytes": avg_bytes,
            "bpp": bpp,
            # Power (all _extract_power keys spread here automatically)
            **power,
            # Energy per tile
            "energy_per_tile_mj": energy_per_tile_mj,
            # Iterations
            "n_warmup": data.get("n_warmup"),
            "n_iters": data.get("n_iters"),
            # Breakdown (nested dict)
            "_step_breakdown": steps,
            # Raw data ref
            "_raw": data,
        }
        rows.append(row)

    df = pd.DataFrame(rows)

    # Ensure consistent ordering
    df["platform"] = pd.Categorical(df["platform"], categories=PLATFORM_ORDER, ordered=True)
    df = df.sort_values(["scenario", "platform"]).reset_index(drop=True)
    if skipped_seq:
        print(
            f"  → Skipped {len(skipped_seq)} sequential file(s) (§11 only): "
            + ", ".join(skipped_seq)
        )
    return df


# ── Load ───────────────────────────────────────────────────────────────────────
df = load_all_results(BENCHMARK_DIR)

print(f"Loaded {len(df)} benchmark results:")
for _, row in df.iterrows():
    pw_str = f"  {row['power_total_w']:.1f} W" if row["power_total_w"] else "  no power"
    print(
        f"  {row['platform']:5s}  {row['scenario']:25s}  {row['latency_ms']:8.2f} ms{pw_str:9s}  ← {row['source_file']}"
    )

# Check completeness
expected = {(p, s) for p in PLATFORM_ORDER for s in SCENARIO_ORDER}
actual = set(zip(df["platform"].astype(str), df["scenario"]))
missing = expected - actual
if missing:
    print(f"\n⚠  Missing {len(missing)} combinations:")
    for p, s in sorted(missing):
        print(f"    {p} / {s}")
else:
    print(f"\n✓ All {len(expected)} (platform x scenario) combinations present.")


## 3 · Overview Table

A compact summary of every `(platform, scenario)` pair.  This is the single source
of truth for all plots below — scan it for anomalies (e.g. a scenario that returned
0 ms latency, or missing power data) before interpreting the charts.

**Reading guide:**
- **Latency** is the mean wall-clock time for one 256x256 patch across `n_iters` iterations,
  measured via `time.perf_counter()` with appropriate synchronisation
  (`torch.cuda.synchronize()` on GPU, blocking VART calls on FPGA).
- **NN / CPU split**: NN = sum of all `gpu_*` / `dpu_*` / `nn_*` steps; CPU = sum of all
  `cpu_*` steps (entropy coding + data manipulation).  The split may not exactly equal
  `latency_ms` due to pre/postprocess steps and rounding.
- **Throughput** = `n_iters / wall_total_s`.  Includes all Python overhead between iterations.
- **Power** columns may be empty if `--power` was not used or if RAPL was unavailable (see §1).

In [ ]:
overview_cols = [
    "scenario",
    "platform",
    "latency_ms",
    "nn_latency_ms",
    "cpu_latency_ms",
    "throughput_fps",
    "power_total_w",
    "power_nn_w",
    "power_cpu_w",
    "energy_per_tile_mj",
    "bpp",
    "n_warmup",
    "n_iters",
]

overview = df[overview_cols].copy()
overview["platform"] = overview["platform"].map(PLATFORM_LABELS)
# Sort by scenario then platform
overview["scenario"] = pd.Categorical(
    overview["scenario"], categories=SCENARIO_ORDER, ordered=True
)
overview = overview.sort_values(["scenario", "platform"]).reset_index(drop=True)

# Format for readability
fmt = {
    "latency_ms": "{:.2f}",
    "nn_latency_ms": "{:.2f}",
    "cpu_latency_ms": "{:.2f}",
    "throughput_fps": "{:.2f}",
    "power_total_w": "{:.2f}",
    "power_nn_w": "{:.2f}",
    "power_cpu_w": "{:.2f}",
    "energy_per_tile_mj": "{:.1f}",
    "bpp": "{:.4f}",
}

overview.style.format(fmt, na_rep="—").set_caption(f"Benchmark overview — {MODEL_NAME}")

## 4 · End-to-End Latency Comparison

**Grouped bar chart**: one cluster per scenario, one bar per platform.

### Interpretation notes

- **What is measured**: Mean per-tile latency (ms) = average of `time.perf_counter()` deltas
  across `n_iters` iterations after `n_warmup` warmup.  Includes all Python overhead between
  timer marks *within* one iteration, but excludes inter-iteration loop overhead.
- **GPU synchronisation**: `torch.cuda.synchronize()` is called before each timer mark,
  ensuring GPU kernels have completed.  Without this, GPU NN steps would appear ~0 ms
  (the kernel launch returns immediately).
- **FPGA synchronisation**: VART `execute_async()` + `wait()` is inherently blocking in
  single-thread mode, so no additional sync is needed.
- **Entropy coding runs on CPU everywhere**: On GPU/CPU platforms, entropy coding uses
  CompressAI's Python + C++ backend on the x86 host.  On FPGA, it uses a C++ `ans.so`
  extension compiled for ARM A53.  The x86 is **much faster** at entropy coding (~30x),
  so total latency differences between GPU and FPGA are dominated by this component, not
  by the NN accelerator.
- **Batch size = 1** on all platforms.  GPU throughput would improve dramatically with
  batching; the FPGA DPU (B4096) only supports batch=1.
- **What is NOT captured**: Inter-tile overhead (tiling, overlap blending), data loading
  from disk/network, host↔device transfer for GPU (negligible for 256x256 FP32), and
  DMA transfer for FPGA (included in VART timing but not separately measured).

In [ ]:
def plot_grouped_bars(
    df: pd.DataFrame,
    metric: str,
    ylabel: str,
    title: str,
    scenarios: Optional[List[str]] = None,
    ax: Optional[Axes] = None,
    log_scale: bool = False,
    bar_label_fmt: str = "{:.1f}",
) -> Axes:
    """Grouped bar chart: one cluster per scenario, one bar per platform."""
    scenarios = scenarios or SCENARIO_ORDER
    platforms = [
        p
        for p in PLATFORM_ORDER
        if p in df["platform"].cat.categories and (df["platform"] == p).any()
    ]

    _own_fig = ax is None
    if _own_fig:
        fig, ax = plt.subplots(figsize=(12, 5))

    x = np.arange(len(scenarios))
    width = 0.8 / len(platforms)

    for i, plat in enumerate(platforms):
        vals = []
        for sc in scenarios:
            row = df[(df["platform"] == plat) & (df["scenario"] == sc)]
            vals.append(row[metric].values[0] if len(row) else 0)
        offset = (i - len(platforms) / 2 + 0.5) * width
        bars = ax.bar(
            x + offset,
            vals,
            width * 0.9,
            label=PLATFORM_LABELS.get(plat, plat),
            color=PLATFORM_COLORS.get(plat, "gray"),
            edgecolor="white",
            linewidth=0.5,
        )
        # Value labels on bars
        for bar, v in zip(bars, vals):
            if v > 0:
                ax.text(
                    bar.get_x() + bar.get_width() / 2,
                    bar.get_height(),
                    bar_label_fmt.format(v),
                    ha="center",
                    va="bottom",
                    fontsize=8,
                )

    ax.set_xticks(x)
    ax.set_xticklabels([SCENARIO_LABELS.get(s, s) for s in scenarios], fontsize=9)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(fontsize=10)
    if log_scale:
        ax.set_yscale("log")
        ax.yaxis.set_major_formatter(mticker.ScalarFormatter())
    if _own_fig:
        fig.tight_layout()
    return ax


plot_grouped_bars(
    df,
    "latency_ms",
    "Latency (ms)",
    "",  # f"Per-patch (256x256) Latency — {MODEL_NAME}",
    log_scale=False,
)
if SAVE_FIGURES:
    PLOTS_DIR.mkdir(parents=True, exist_ok=True)
    plt.savefig(
        PLOTS_DIR / f"latency_comparison_all_scenarios.{FIGURE_FORMAT}", bbox_inches="tight"
    )
plt.show()

# Print a table where for each scenario we see the speed-up factor, from FPGA to CPU and FPGA to GPU.
speedup_rows = []
for sc in SCENARIO_ORDER:
    fpga_row = df[(df["platform"] == "fpga") & (df["scenario"] == sc)]
    gpu_row = df[(df["platform"] == "gpu") & (df["scenario"] == sc)]
    cpu_row = df[(df["platform"] == "cpu") & (df["scenario"] == sc)]
    if fpga_row.empty:
        continue
    fpga_ms = fpga_row.iloc[0]["latency_ms"]
    gpu_ms = gpu_row.iloc[0]["latency_ms"] if not gpu_row.empty else None
    cpu_ms = cpu_row.iloc[0]["latency_ms"] if not cpu_row.empty else None
    speedup_rows.append(
        {
            "Scenario": SCENARIO_LABELS.get(sc, sc),
            "FPGA (ms)": round(fpga_ms, 2),
            "CPU (ms)": round(cpu_ms, 2) if cpu_ms else None,
            "GPU (ms)": round(gpu_ms, 2) if gpu_ms else None,
            "Speed-up FPGA→CPU": round(fpga_ms / cpu_ms, 2) if cpu_ms else None,
            "Speed-up FPGA→GPU": round(fpga_ms / gpu_ms, 2) if gpu_ms else None,
        }
    )

speedup_df = pd.DataFrame(speedup_rows).set_index("Scenario")
speedup_df.style.format(
    {
        "FPGA (ms)": "{:.2f}",
        "CPU (ms)": "{:.2f}",
        "GPU (ms)": "{:.2f}",
        "Speed-up FPGA→CPU": "{:.2f}×",
        "Speed-up FPGA→GPU": "{:.2f}×",
    },
    na_rep="—",
).set_caption(f"Speed-up factors (FPGA as baseline) — {MODEL_NAME}")


## 5 · Per-Step Latency Breakdown (Stacked Bars)

Shows **where time is spent** within each scenario on each platform.  Steps are grouped
into two categories:

- **NN steps** (`nn_g_a`, `nn_h_a`, `nn_h_s`, `nn_g_s`): model forward passes on the
  accelerator (GPU CUDA cores / FPGA DPU / CPU via PyTorch).
- **CPU steps** (`cpu_eb_compress`, `cpu_eb_decompress`, `cpu_gc_compress`,
  `cpu_gc_decompress`, `cpu_concat_abs`, `cpu_split_y_hat`): entropy coding and data
  manipulation, always on CPU (x86 or ARM A53).

### Interpretation notes

- Step labels have been **canonicalised**: `gpu_g_a` → `nn_g_a`, `dpu_g_a` → `nn_g_a`.
  This allows direct visual comparison of the same logical step across platforms.
- **GC compress/decompress dominates** on all platforms because the Gaussian Conditional
  entropy coder operates on 256-channel, 16x16 tensors (~65K symbols) using sequential
  rANS.  On the ARM A53, this is 10–30x slower than on x86.
- **NN steps on FPGA include Python/VART overhead**: The FPGA NN times include numpy
  INT8 quantisation, VART dispatch, and numpy dequantisation — typically adding 10–20%
  over the raw DPU hardware time (see `xdputil benchmark` comparison in the doc §6.4).

In [ ]:
# ── Colour palette for individual steps ────────────────────────────────────────
STEP_COLORS = {
    # NN subgraphs (blue shades — from plots_colors.json latency_steps)
    "nn_g_a": C["latency_steps"]["nn_g_a"],
    "nn_h_a": C["latency_steps"]["nn_h_a"],
    "nn_h_s": C["latency_steps"]["nn_h_s"],
    "nn_g_s": C["latency_steps"]["nn_g_s"],
    # CPU entropy steps (orange/red shades)
    "cpu_eb_compress": C["latency_steps"]["cpu_eb_compress"],
    "cpu_eb_decompress": C["latency_steps"]["cpu_eb_decompress"],
    "cpu_gc_compress": C["latency_steps"]["cpu_gc_compress"],
    "cpu_gc_decompress": C["latency_steps"]["cpu_gc_decompress"],
    # CPU data ops (grey shades)
    "cpu_concat_abs": C["latency_steps"]["cpu_concat_abs"],
    "cpu_split_y_hat": C["latency_steps"]["cpu_split_y_hat"],
}

# Preferred step order (bottom → top in stacked bar)
STEP_ORDER = [
    "nn_g_a",
    "cpu_concat_abs",
    "nn_h_a",
    "cpu_eb_compress",
    "cpu_eb_decompress",
    "nn_h_s",
    "cpu_gc_compress",
    "cpu_gc_decompress",
    "cpu_split_y_hat",
    "nn_g_s",
]


def plot_step_breakdown(
    df: pd.DataFrame,
    scenario: str,
    ax: Optional[Axes] = None,
    min_ms: float = 0.1,
) -> Axes:
    """Stacked horizontal bar chart for one scenario across all platforms."""
    subset = df[df["scenario"] == scenario].sort_values("platform")
    if subset.empty:
        raise ValueError(f"No data for scenario '{scenario}'")

    _own = ax is None
    if _own:
        fig, ax = plt.subplots(figsize=(12, max(2, len(subset) * 1.2)))

    y_labels = []
    for idx, (_, row) in enumerate(subset.iterrows()):
        plat = str(row["platform"])
        steps: Dict[str, float] = row["_step_breakdown"]
        y_labels.append(PLATFORM_LABELS.get(plat, plat))

        # Order steps, skip tiny ones
        ordered = [(s, steps.get(s, 0)) for s in STEP_ORDER if steps.get(s, 0) >= min_ms]
        # Add any remaining steps not in STEP_ORDER
        known = set(STEP_ORDER)
        for s, v in sorted(steps.items()):
            if s not in known and v >= min_ms and not s.startswith("_"):
                ordered.append((s, v))

        left = 0.0
        for step_name, ms_val in ordered:
            color = STEP_COLORS.get(step_name, "#bdc3c7")
            bar = ax.barh(
                idx,
                ms_val,
                left=left,
                height=0.6,
                color=color,
                edgecolor="white",
                linewidth=0.5,
                label=step_name,
            )
            # Label if wide enough
            if ms_val > row["latency_ms"] * 0.05:
                ax.text(
                    left + ms_val / 2,
                    idx,
                    f"{ms_val:.1f}",
                    ha="center",
                    va="center",
                    fontsize=6.5,
                    color="white",
                    fontweight="bold",
                )
            left += ms_val

    ax.set_yticks(range(len(y_labels)))
    ax.set_yticklabels(y_labels)
    ax.set_xlabel("Latency (ms)")
    ax.set_title(
        f"Step Breakdown — {SCENARIO_LABELS.get(scenario, scenario).replace(chr(10), ' ')} — {MODEL_NAME}"
    )
    ax.invert_yaxis()

    # Deduplicated legend — sorted by STEP_ORDER so legend always matches bar order,
    # regardless of which platform first registered each label with matplotlib.
    handles, labels = ax.get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    ordered_keys = [k for k in STEP_ORDER if k in by_label]
    ordered_keys += [k for k in by_label if k not in set(STEP_ORDER)]  # extras at end
    ax.legend(
        [by_label[k] for k in ordered_keys],
        ordered_keys,
        loc="upper right",
        fontsize=7,
        ncol=2,
    )

    if _own:
        fig.tight_layout()
    return ax


# Plot the "full" scenario breakdown (the most informative)
scenario_plot = "compress"
plot_step_breakdown(df, scenario_plot)
if SAVE_FIGURES:
    plt.savefig(PLOTS_DIR / f"step_breakdown_{scenario_plot}.{FIGURE_FORMAT}", bbox_inches="tight")
plt.show()


In [ ]:
# Breakdown for all other scenarios (compact 2x2 grid)
other_scenarios = [s for s in SCENARIO_ORDER if s != "full" and s in df["scenario"].values]
if other_scenarios:
    n = len(other_scenarios)
    ncols = min(2, n)
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(14, 3.5 * nrows))
    axes_flat = np.array(axes).flatten() if n > 1 else [axes]
    for i, sc in enumerate(other_scenarios):
        plot_step_breakdown(df, sc, ax=axes_flat[i])
    for j in range(i + 1, len(axes_flat)):
        axes_flat[j].set_visible(False)
    fig.tight_layout()
    if SAVE_FIGURES:
        plt.savefig(PLOTS_DIR / f"step_breakdown_all.{FIGURE_FORMAT}", bbox_inches="tight")
    plt.show()

## 5b · NN vs CPU Latency Split (simplified step breakdown)

Same scenario as §5 but collapsed into two colours:
* **NN / DPU** — all steps whose name starts with `nn_`
* **CPU** — all other steps

Adjacent steps of the same type are merged into one bar segment, matching the
physical execution ports (e.g. `cpu_eb_compress` + `cpu_eb_decompress` become a
single CPU block because they sit back-to-back on the CPU between two NN subgraphs).

Use `label_min_ms=-1` to suppress all in-bar labels.

In [ ]:
NN_COLOR = C["nn_vs_cpu_domain"]["nn_dynamic"]  # blue — DPU / NN subgraph
CPU_COLOR = C["nn_vs_cpu_domain"]["cpu_dynamic"]  # orange — CPU work


def _classify_step(name: str) -> str:
    """Return 'nn' if the step runs on the DPU/NN, 'cpu' otherwise."""
    return "nn" if name.startswith("nn_") else "cpu"


def _merge_adjacent(ordered: List[Tuple[str, float]]) -> List[Tuple[str, float]]:
    """Collapse consecutive same-type steps into (type, total_ms) pairs."""
    merged: List[Tuple[str, float]] = []
    for step_name, ms_val in ordered:
        kind = _classify_step(step_name)
        if merged and merged[-1][0] == kind:
            merged[-1] = (kind, merged[-1][1] + ms_val)
        else:
            merged.append((kind, ms_val))
    return merged


def plot_nn_cpu_breakdown(
    df: pd.DataFrame,
    scenario: str,
    ax: Optional[Axes] = None,
    min_ms: float = 0.1,
    label_min_ms: float = 5.0,
    with_title: bool = True,
) -> Axes:
    """Simplified stacked horizontal bar chart: NN vs CPU blocks only.

    Adjacent steps that belong to the same execution port (NN or CPU) are
    merged into a single coloured segment.

    Parameters
    ----------
    df:
        DataFrame produced by the data-loading cell (must contain
        ``_step_breakdown`` and ``latency_ms`` columns).
    scenario:
        One of the scenario keys, e.g. ``"full"``, ``"compress"``.
    ax:
        Existing :class:`~matplotlib.axes.Axes` to draw on.  A new figure is
        created when *None*.
    min_ms:
        Steps below this threshold (ms) are skipped entirely.
    label_min_ms:
        Minimum segment width (ms) at which the numeric label is drawn inside
        the bar.  Pass ``-1`` (or any negative value) to suppress all labels.
    """
    subset = df[df["scenario"] == scenario].sort_values("platform")
    if subset.empty:
        raise ValueError(f"No data for scenario '{scenario}'")

    if ax is None:
        fig, ax = plt.subplots(figsize=(12, max(2, len(subset) * 1.2)))

    y_labels = []
    for idx, (_, row) in enumerate(subset.iterrows()):
        plat = str(row["platform"])
        steps: Dict[str, float] = row["_step_breakdown"]
        y_labels.append(PLATFORM_LABELS.get(plat, plat))

        # Collect steps in canonical order, dropping tiny ones
        ordered: List[Tuple[str, float]] = [
            (s, steps.get(s, 0.0)) for s in STEP_ORDER if steps.get(s, 0.0) >= min_ms
        ]
        # Append any steps not listed in STEP_ORDER
        known = set(STEP_ORDER)
        for s, v in sorted(steps.items()):
            if s not in known and v >= min_ms and not s.startswith("_"):
                ordered.append((s, v))

        # Merge consecutive same-type steps into one segment
        merged = _merge_adjacent(ordered)

        left = 0.0
        _nn_labelled = _cpu_labelled = False  # legend: register each colour once
        for kind, ms_val in merged:
            color = NN_COLOR if kind == "nn" else CPU_COLOR
            label = "NN / DPU" if kind == "nn" else "CPU"
            # Only register the label the first time (for dedup legend)
            use_label = label
            if kind == "nn" and _nn_labelled:
                use_label = "_nolegend_"
            elif kind == "cpu" and _cpu_labelled:
                use_label = "_nolegend_"

            ax.barh(
                idx,
                ms_val,
                left=left,
                height=0.8,
                color=color,
                edgecolor="white",
                linewidth=0.8,
                label=use_label,
            )

            # In-bar numeric label
            if label_min_ms >= 0 and ms_val >= label_min_ms:
                ax.text(
                    left + ms_val / 2,
                    idx,
                    f"{ms_val:.1f}",
                    ha="center",
                    va="center",
                    fontsize=8.5,
                    color="white",
                    fontweight="bold",
                )

            if kind == "nn":
                _nn_labelled = True
            else:
                _cpu_labelled = True

            left += ms_val

    ax.set_yticks(range(len(y_labels)))
    ax.set_yticklabels(y_labels)
    ax.set_xlabel("Latency (ms)")
    if with_title:
        ax.set_title(
            f"NN vs CPU Breakdown — {SCENARIO_LABELS.get(scenario, scenario)} — {MODEL_NAME}"
        )
    ax.invert_yaxis()

    handles, labels = ax.get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    ax.legend(list(by_label.values()), list(by_label.keys()), loc="upper right", fontsize=12)

    return ax


# ── Example: Full scenario ─────────────────────────────────────────────────────
plot_nn_cpu_breakdown(df, "compress", label_min_ms=5.0, with_title=False)
if SAVE_FIGURES:
    plt.savefig(
        PLOTS_DIR / f"latencies_nn_cpu_breakdown_full.{FIGURE_FORMAT}", bbox_inches="tight"
    )
plt.show()


## 6 · NN vs Entropy Latency Split

This chart isolates the **NN accelerator time** from the **CPU entropy time** for each
`(platform, scenario)`.  It directly answers: *"Is the bottleneck in the neural network
or in the entropy coder?"*

### Interpretation notes

- **NN time** = sum of all steps with `gpu_` / `dpu_` / `nn_` prefix.
  On GPU this is the time CUDA kernels spend executing the four subgraphs (g_a, h_a, h_s, g_s).
  On FPGA this is the time the DPU spends + Python/VART overhead.
  On CPU this is PyTorch CPU forward passes.
- **CPU time** = sum of all steps with `cpu_` prefix.
  Entropy coding (EB compress/decompress + GC compress/decompress) + data manipulation
  (concat, abs, split).  This runs on x86 for GPU/CPU benchmarks and on ARM A53 for FPGA.
- **Key insight for deployment**: If CPU time >> NN time, optimising the NN accelerator
  (more DPU cores, higher frequency, GPU upgrade) yields negligible speedup.  Instead,
  the entropy coder must be optimised (faster rANS, hardware entropy coder, or eliminating
  entropy coding from the latency-critical path via pre-computed codebooks).
- **The `nn_only` and `entropy_only` scenarios** should show near-zero in their complementary
  bar — they exist precisely to isolate one component.

**Formulas** (from the JSON output):
$$t_\text{NN} = \sum_{s \in \text{nn\_*}} s.\text{mean\_s} \times 1000 \quad\text{(ms)}$$
$$t_\text{CPU} = \sum_{s \in \text{cpu\_*}} s.\text{mean\_s} \times 1000 \quad\text{(ms)}$$
$$t_\text{total} \approx t_\text{NN} + t_\text{CPU} + t_\text{overhead}$$

where $t_\text{overhead}$ captures pre/postprocess and timer overhead (typically < 0.1 ms).

In [ ]:
def plot_nn_vs_cpu_split(df: pd.DataFrame, scenarios: Optional[List[str]] = None) -> None:
    """Side-by-side grouped bars showing NN vs CPU time for each platform x scenario."""
    scenarios = scenarios or ["full", "compress", "decompress"]
    fig, axes = plt.subplots(1, len(scenarios), figsize=(5 * len(scenarios), 5), sharey=True)
    if len(scenarios) == 1:
        axes = [axes]

    for ax, sc in zip(axes, scenarios):
        subset = df[df["scenario"] == sc].sort_values("platform")
        plats = subset["platform"].astype(str).tolist()
        nn_vals = subset["nn_latency_ms"].values
        cpu_vals = subset["cpu_latency_ms"].values

        x = np.arange(len(plats))
        w = 0.35
        bars_nn = ax.bar(
            x - w / 2, nn_vals, w, label="NN (accelerator)", color=NN_COLOR, edgecolor="white"
        )
        bars_cpu = ax.bar(
            x + w / 2,
            cpu_vals,
            w,
            label="CPU (entropy + data)",
            color=CPU_COLOR,
            edgecolor="white",
        )

        for bar, v in zip(bars_nn, nn_vals):
            if v > 0:
                ax.text(
                    bar.get_x() + bar.get_width() / 2,
                    bar.get_height(),
                    f"{v:.1f}",
                    ha="center",
                    va="bottom",
                    fontsize=7,
                )
        for bar, v in zip(bars_cpu, cpu_vals):
            if v > 0:
                ax.text(
                    bar.get_x() + bar.get_width() / 2,
                    bar.get_height(),
                    f"{v:.1f}",
                    ha="center",
                    va="bottom",
                    fontsize=7,
                )

        ax.set_xticks(x)
        ax.set_xticklabels([PLATFORM_LABELS.get(p, p) for p in plats], fontsize=8, rotation=15)
        ax.set_title(SCENARIO_LABELS.get(sc, sc).replace("\n", " "))
        ax.set_ylabel("Latency (ms)" if ax == axes[0] else "")
        ax.legend(fontsize=7, loc="upper left")

    fig.suptitle(f"NN vs CPU Latency Split — {MODEL_NAME}", fontsize=12, y=1.02)
    fig.tight_layout()
    if SAVE_FIGURES:
        plt.savefig(PLOTS_DIR / f"nn_vs_cpu_split.{FIGURE_FORMAT}", bbox_inches="tight")
    plt.show()


plot_nn_vs_cpu_split(df)


In [ ]:
# Pie charts: NN vs CPU fraction for the "full" scenario on each platform
scenario = "compress"
scenario_df = df[df["scenario"] == scenario].sort_values("platform")
if not scenario_df.empty:
    fig, axes = plt.subplots(1, len(scenario_df), figsize=(4.5 * len(scenario_df), 4))
    if len(scenario_df) == 1:
        axes = [axes]
    for ax, (_, row) in zip(axes, scenario_df.iterrows()):
        nn = row["nn_latency_ms"]
        cpu = row["cpu_latency_ms"]
        other = max(0, row["latency_ms"] - nn - cpu)
        sizes = [nn, cpu] + ([other] if other > 0.5 else [])
        labels = [f"NN\n{nn:.1f} ms", f"CPU\n{cpu:.1f} ms"]
        colors = [NN_COLOR, CPU_COLOR]
        if other > 0.5:
            labels.append(f"Other\n{other:.1f} ms")
            colors.append("#bdc3c7")
        ax.pie(
            sizes,
            labels=labels,
            colors=colors,
            autopct="%1.0f%%",
            startangle=90,
            textprops={"fontsize": 8},
        )
        ax.set_title(PLATFORM_LABELS.get(str(row["platform"]), str(row["platform"])), fontsize=10)
    fig.suptitle(f"Latency Composition ({scenario} scenario) — {MODEL_NAME}", fontsize=12, y=1.02)
    fig.tight_layout()
    if SAVE_FIGURES:
        plt.savefig(PLOTS_DIR / f"nn_cpu_pie.{FIGURE_FORMAT}", bbox_inches="tight")
    plt.show()


## 7 · Throughput Comparison

**Throughput** (patches/s) = `n_iters / wall_total_s`, where `wall_total_s` is the
wall-clock time for all measured iterations (excludes warmup).

### Interpretation notes

- Throughput is the **inverse of latency** only if there is no inter-iteration overhead.
  In practice, the Python loop adds a few µs per iteration, so throughput ≈ 1000 / latency_ms.
- **Batch=1 everywhere**: GPU throughput would increase significantly with larger batches
  (the GPU is severely underutilised at batch=1 for this small model).  FPGA DPU B4096
  only supports batch=1.
- **`nn_only` throughput** is the theoretical maximum if entropy coding were instantaneous
  (e.g. via a hardware entropy accelerator or if we don't need entropy coding at all).
- Throughput is useful for **system sizing**: "How many patches per second can this
  platform process?" — e.g. for real-time SAR processing pipelines.

$$\text{Throughput} = \frac{N_\text{iters}}{t_\text{wall}} \;\text{(patches/s)}$$

In [ ]:
plot_grouped_bars(
    df,
    "throughput_fps",
    "Throughput (patches/s)",
    "",  # f"Throughput — {MODEL_NAME}",
    log_scale=False,
    bar_label_fmt="{:.1f}",
)
if SAVE_FIGURES:
    plt.savefig(PLOTS_DIR / f"throughput_comparison.{FIGURE_FORMAT}", bbox_inches="tight")
plt.show()

## 8 · Power Consumption

### Measurement scope — read carefully before citing any number

Each platform uses a fundamentally different measurement technique:

| Platform | Technique | What IS measured | What is NOT measured |
|---|---|---|---|
| **FPGA** | INA226 (18 rails, ~50 Hz) + PMBus (3 rails) | PL fabric, PS (ARM A53), BRAM, DDR I/O, UTIL rails — **TBP ≈ 10.9–18.7 W** | DDR DRAM cells (only I/O), SD card, USB, HDMI, barrel jack losses |
| **GPU** | `nvidia-smi` (10 Hz) | GPU board power: die + VRAM + VRMs | Host CPU, motherboard, PSU, fans, RAM |
| **CPU** | Intel RAPL (via sysfs) | x86 CPU package (all cores + uncore) | Motherboard, PSU, GPU idle draw, fans, NVMe |

**⚠ Key caveats:**

1. **Scopes are incomparable**: FPGA `TBP` ≈ 11–19 W covers the whole ZCU102 SoC+peripherals.
   GPU `power_gpu` ≈ 50–70 W covers only the GPU board; the host CPU simultaneously draws
   another 52–77 W (RAPL) during entropy coding — but that CPU is absent in an FPGA deployment.
2. **Idle power is large**: FPGA idle TBP ≈ 10.9 W; GPU idle ≈ 33–34 W (CUDA context loaded).
   **Dynamic power** = load − idle represents the *incremental cost* of running one inference
   and is the most deployment-relevant metric.
3. **nvidia-smi resolution**: The driver reports a smoothed reading at ~10 Hz internally.
   Short workloads (< 500 ms wall time) will have few samples; interpret with caution.
4. **RAPL requires root** (Linux ≥ 5.10): If `idle_cpu_rapl` is missing from a result file,
   fix with `sudo chmod o+r /sys/class/powercap/intel-rapl/*/energy_uj`.
5. **INA226 lower bound**: Six rails are dark (N/A in UG1182 Table 3-55); estimated < 500 mW
   total, workload-invariant. See §4.8 and §8.1 of `FPGA_benchmark.md`.

### Power group definitions (FPGA)

| Group | Rails | Role |
|---|---|---|
| **PL** | VCCINT + VCCBRAM + VCCAUX + VCC1V2 + VCC3V3 | FPGA fabric — DPU switching power lives here |
| **PS** | VCCPSINTFP + VCCPSINTLP + VCCPSAUX + VCCPSPLL + VCCO_PSDDR_504 + VCCOPS + VCCOPS3 + VCCPSDDRPLL | ARM A53 quad-core + DDR I/O |
| **MGT** | MGTAVCC + MGTAVTT + MGTRAVCC + MGTRAVTT | SerDes transceivers — unused by DPU, ~0.11 W constant |
| **Peripherals** | DDR4_DIMM_VDDQ + UTIL_3V3 + UTIL_5V0 (PMBus) | Board peripherals |
| **MPSoC** | PL + PS | SoC compute boundary |
| **TBP** | MPSoC + MGT + Peripherals | Most complete FPGA board power from sensors |

### Sub-group interpretation

- **DPU_fabric** = VCCINT + VCCBRAM: captures FPGA LUT/FF switching — the actual NN workload.
- **PS_compute** = VCCPSINTFP + VCCPSINTLP: captures ARM A53 core activity — entropy coding.

$$P_{\text{dynamic}} = P_{\text{load}} - P_{\text{idle}}$$

### Plots in this section

- **§8a** FPGA power composition by scenario: stacked PL / PS / MGT / Peripherals.
  Dashed line = idle TBP (≈ 10.9 W across all scenarios; small variation is measurement noise).
- **§8b** GPU power by scenario: GPU board and CPU RAPL bars each split into idle (pale) +
  dynamic (solid). Shows that CPU RAPL dominates for entropy-heavy scenarios.
- **§8c** Dynamic power comparison: incremental cost per inference across platforms.
  ⚠ FPGA `TBP_dynamic` and `GPU_dynamic` have different scopes — do not subtract to compare
  efficiency without accounting for deployment context.
- **§8d** FPGA DPU_fabric vs PS_compute dynamic: reveals which silicon domain drives the
  power increase for each scenario type. PS_compute increase confirms ARM A53 entropy load.


In [ ]:
# ── §8a  FPGA Power Composition ───────────────────────────────────────────────
# Stacked load bars: PL | PS | Peripherals by scenario.
# MGT (~0.11 W, transceivers unused) and FMC (0.0 W, slot unused) are excluded
# from the visual stack but ARE included in the TBP total shown on top.
#
# Reading guide:
#   Gap between a group's load bar top and its idle line = dynamic power.
#   PL varies widely (DPU switching); PS and Peripherals are nearly constant.


def plot_fpga_power_composition(df: pd.DataFrame) -> None:
    """Stacked bar: FPGA TBP by power group across all measured scenarios.

    Layers (bottom to top): PL | PS | Peripherals.
    MGT (~0.11 W constant) and FMC (0.0 W) excluded from visual bars but
    included in TBP total annotation (small gap at top = MGT + FMC).

    Idle threshold lines per cumulative group boundary:
      idle PL   (solid)  — PL idle level
      idle PL+PS (dashed) — cumulative PL+PS idle
      idle TBP  (dashed red) — full board idle
    """
    fpga = df[(df["platform"] == "fpga") & df["power_TBP_w"].notna()].copy()
    if fpga.empty:
        print("No FPGA power data available.")
        return

    scenarios = [s for s in SCENARIO_ORDER if s in fpga["scenario"].values]
    scenario_lbl = [SCENARIO_LABELS.get(s, s).replace("\n", " ") for s in scenarios]
    n = len(scenarios)
    x = np.arange(n)

    # MGT excluded from bars (constant ~0.11 W, transceivers unused; included in TBP total)
    STACK_COLS = ["power_PL_w", "power_PS_w", "power_peripherals_w"]
    STACK_LABELS = [
        "PL  (VCCINT + BRAM + AUX + 1V2 + 3V3)",
        "PS  (ARM A53 + DDR I/O + PLLs)",
        "Peripherals  (DDR4 DIMM + UTIL rails)",
    ]
    STACK_COLORS = [
        C["fpga_power_groups"]["PL"],
        C["fpga_power_groups"]["PS"],
        C["fpga_power_groups"]["peripherals"],
    ]

    fig, ax = plt.subplots(figsize=(11, 5.5))
    bottoms = np.zeros(n)

    for col, lbl, color in zip(STACK_COLS, STACK_LABELS, STACK_COLORS):
        vals = np.array(
            [
                float(fpga.loc[fpga["scenario"] == s, col].values[0])
                if col in fpga.columns and len(fpga[fpga["scenario"] == s]) > 0
                else 0.0
                for s in scenarios
            ]
        )
        ax.bar(
            x,
            vals,
            bottom=bottoms,
            label=lbl,
            color=color,
            edgecolor="white",
            linewidth=0.6,
            width=0.62,
        )
        for xi, v, b in zip(x, vals, bottoms):
            if v > 0.25:
                ax.text(
                    xi,
                    b + v / 2,
                    f"{v:.2f}",
                    ha="center",
                    va="center",
                    fontsize=7.5,
                    color="white",
                    fontweight="bold",
                )
        bottoms += vals

    # TBP total annotation on top (includes MGT + FMC not shown in bars)
    for xi, s in enumerate(scenarios):
        row = fpga[fpga["scenario"] == s]
        tbp = float(row["power_TBP_w"].values[0]) if not row.empty else 0.0
        ax.text(
            xi,
            tbp + 0.15,
            f"{tbp:.2f} W",
            ha="center",
            va="bottom",
            fontsize=8.5,
            fontweight="bold",
        )

    # ── Per-group cumulative idle threshold lines ──────────────────────────────
    def _mean_idle(col_or_cols):
        """Mean across scenarios of a cumulative idle value."""
        vals = []
        for s in scenarios:
            row = fpga[fpga["scenario"] == s]
            if row.empty:
                continue
            if isinstance(col_or_cols, list):
                v = sum(
                    float(row[c].values[0])
                    for c in col_or_cols
                    if c in row.columns and not pd.isna(row[c].values[0])
                )
            else:
                c = col_or_cols
                if c not in row.columns or pd.isna(row[c].values[0]):
                    continue
                v = float(row[c].values[0])
            vals.append(v)
        return float(np.mean(vals)) if vals else None

    threshold_styles = [
        ("idle_PL_w", C["fpga_power_groups"]["PL"], "solid", 1.4, "idle PL"),
        (["idle_PL_w", "idle_PS_w"], C["fpga_power_groups"]["PS"], "dashed", 1.4, "idle PL+PS"),
        ("idle_TBP_w", C["fpga_power_groups"]["TBP_line"], "dashed", 1.8, "idle TBP"),
    ]
    for col_or_cols, color, ls, lw, label in threshold_styles:
        level = _mean_idle(col_or_cols)
        if level is None:
            continue
        ax.axhline(level, color=color, linestyle=ls, linewidth=lw, alpha=0.85, zorder=3)
        ax.text(
            n - 0.28, level + 0.10, label, color="black", fontsize=7.5, ha="right", va="bottom"
        )

    ax.set_xticks(x)
    ax.set_xticklabels(scenario_lbl, fontsize=9)
    ax.set_ylabel("Average Power  (W)")
    ax.set_title(
        f"FPGA (ZCU102) Power Composition by Scenario — {MODEL_NAME}\n"
        "Note: MGT (~0.11 W) and FMC (0.0 W) excluded from bars, included in TBP total"
    )
    ax.legend(loc="upper left", fontsize=8, ncol=2)
    ax.set_ylim(bottom=0)
    fig.tight_layout()
    if SAVE_FIGURES:
        plt.savefig(PLOTS_DIR / f"power_fpga_composition.{FIGURE_FORMAT}", bbox_inches="tight")
    plt.show()


plot_fpga_power_composition(df)


# ── §8b  GPU system + CPU platform power breakdown ────────────────────────────
# Left bar  : GPU system = GPU board (NN domain) stacked with CPU RAPL (entropy/OS).
#             Both split into idle (pale) + dynamic (solid).
#             Comparable scope to FPGA TBP: GPU board ≈ DPU fabric, RAPL ≈ PS_compute.
# Right bar : CPU-only platform RAPL (separate benchmark run; not concurrent with GPU).


def plot_gpu_power_breakdown(df: pd.DataFrame) -> None:
    """GPU system stacked bar + CPU-only reference bar per scenario.

    GPU system = GPU board (bottom) stacked with CPU RAPL (top).
    Each domain split into idle (pale) and dynamic (solid) components.
    Value labels inside each segment; total on top.
    """
    gpu = df[(df["platform"] == "gpu") & df["power_gpu_w"].notna()].copy()
    cpu = df[(df["platform"] == "cpu") & df["power_total_w"].notna()].copy()

    all_sc = [
        s for s in SCENARIO_ORDER if s in gpu["scenario"].values or s in cpu["scenario"].values
    ]
    if not all_sc:
        print("No GPU/CPU power data available.")
        return

    n = len(all_sc)
    x = np.arange(n)
    w = 0.35

    def _get(frame: pd.DataFrame, col: str, sc: str) -> float:
        row = frame[frame["scenario"] == sc]
        if row.empty or col not in row.columns:
            return 0.0
        v = row[col].values[0]
        return float(v) if v is not None and not pd.isna(v) else 0.0

    gpu_idle = np.array([_get(gpu, "idle_gpu_w", s) for s in all_sc])
    gpu_load = np.array([_get(gpu, "power_gpu_w", s) for s in all_sc])
    rapl_idle = np.array([_get(gpu, "idle_cpu_rapl_w", s) for s in all_sc])
    rapl_load = np.array([_get(gpu, "power_cpu_rapl_w", s) for s in all_sc])
    gpu_dyn = np.maximum(0.0, gpu_load - gpu_idle)
    rapl_dyn = np.maximum(0.0, rapl_load - rapl_idle)

    cpu_idle_v = np.array([_get(cpu, "power_idle_total_w", s) for s in all_sc])
    cpu_load_v = np.array([_get(cpu, "power_total_w", s) for s in all_sc])
    cpu_dyn_v = np.maximum(0.0, cpu_load_v - cpu_idle_v)

    GB_IDLE = C["gpu_power_domains"]["gpu_board_idle"]
    GB_DYN = C["gpu_power_domains"]["gpu_board_dynamic"]
    RL_IDLE = C["gpu_power_domains"]["cpu_rapl_idle"]
    RL_DYN = C["gpu_power_domains"]["cpu_rapl_dynamic"]
    CP_IDLE = C["platforms"]["cpu_idle"]
    CP_DYN = C["platforms"]["cpu_dynamic"]

    fig, ax = plt.subplots(figsize=(11, 5.5))

    # ── GPU system: GPU board (idle+dyn) stacked with CPU RAPL (idle+dyn) ──
    ax.bar(x - w / 2, gpu_idle, w, color=GB_IDLE, edgecolor="white", label="GPU board — idle")
    ax.bar(
        x - w / 2,
        gpu_dyn,
        w,
        bottom=gpu_idle,
        color=GB_DYN,
        edgecolor="white",
        label="GPU board — dynamic",
    )
    gpu_total = gpu_idle + gpu_dyn
    ax.bar(
        x - w / 2,
        rapl_idle,
        w,
        bottom=gpu_total,
        color=RL_IDLE,
        edgecolor="white",
        label="CPU RAPL — idle",
    )
    ax.bar(
        x - w / 2,
        rapl_dyn,
        w,
        bottom=gpu_total + rapl_idle,
        color=RL_DYN,
        edgecolor="white",
        label="CPU RAPL — dynamic",
    )

    # ── CPU-only platform ──
    ax.bar(x + w / 2, cpu_idle_v, w, color=CP_IDLE, edgecolor="white", label="CPU only — idle")
    ax.bar(
        x + w / 2,
        cpu_dyn_v,
        w,
        bottom=cpu_idle_v,
        color=CP_DYN,
        edgecolor="white",
        label="CPU only — dynamic",
    )

    # ── Inline segment labels + total on top ──
    MIN_LABEL_W = 5.0
    for xi, gi, gd, ri, rd, ci, cd in zip(
        x, gpu_idle, gpu_dyn, rapl_idle, rapl_dyn, cpu_idle_v, cpu_dyn_v
    ):
        for ybot, height in [(0, gi), (gi, gd), (gi + gd, ri), (gi + gd + ri, rd)]:
            if height >= MIN_LABEL_W:
                ax.text(
                    xi - w / 2,
                    ybot + height / 2,
                    f"{height:.0f}",
                    ha="center",
                    va="center",
                    fontsize=7,
                    color="white",
                    fontweight="bold",
                )
        sys_total = gi + gd + ri + rd
        if sys_total > 0:
            ax.text(
                xi - w / 2,
                sys_total + 0.8,
                f"{sys_total:.0f} W",
                ha="center",
                va="bottom",
                fontsize=8,
                fontweight="bold",
            )
        for ybot, height in [(0, ci), (ci, cd)]:
            if height >= MIN_LABEL_W:
                ax.text(
                    xi + w / 2,
                    ybot + height / 2,
                    f"{height:.0f}",
                    ha="center",
                    va="center",
                    fontsize=7,
                    color="white",
                    fontweight="bold",
                )
        cpu_total = ci + cd
        if cpu_total > 0:
            ax.text(
                xi + w / 2,
                cpu_total + 0.8,
                f"{cpu_total:.0f} W",
                ha="center",
                va="bottom",
                fontsize=8,
                fontweight="bold",
            )

    ax.set_xticks(x)
    ax.set_xticklabels([SCENARIO_LABELS.get(s, s).replace("\n", " ") for s in all_sc], fontsize=9)
    ax.set_ylabel("Average Power  (W)")
    ax.set_title(
        f"GPU System + CPU Platform Power — {MODEL_NAME}\n"
        "Left: GPU system (GPU board + CPU RAPL concurrent)  ·  Right: CPU-only platform"
    )
    ax.legend(fontsize=8, ncol=2, loc="upper right")
    ax.set_ylim(bottom=0)
    fig.tight_layout()
    if SAVE_FIGURES:
        plt.savefig(PLOTS_DIR / f"power_gpu_breakdown.{FIGURE_FORMAT}", bbox_inches="tight")
    plt.show()


plot_gpu_power_breakdown(df)


In [ ]:
# ── §8c  Dynamic Power Comparison: FPGA TBP vs GPU board vs CPU RAPL ──────────
# Dynamic = load − idle.  This is the *incremental* cost of running one inference.
# Idle static power (leakage, clock trees, always-on logic) is subtracted.
#
# Shown bars:
#   FPGA TBP dynamic  — full SoC+peripherals incremental cost (INA226 + PMBus)
#   GPU  board dynamic — GPU die+VRAM+VRM incremental cost (nvidia-smi)
#   CPU  RAPL dynamic  — x86 package incremental cost (shown for reference)
#
# ⚠ Scope warning: FPGA and GPU dynamic cover different physical components.
#   FPGA dynamic ≈ SoC ΔP (useful for on-board power budget on a satellite).
#   GPU dynamic ≈ GPU board ΔP (relevant for data-centre power billing per job).
#   CPU RAPL dynamic is additive to GPU dynamic in a GPU-hosted deployment.


def plot_dynamic_power_comparison(df: pd.DataFrame) -> None:
    """Grouped bar chart of dynamic power (load − idle) per platform and scenario."""
    PLATFORM_SPECS = {
        "fpga": (
            "power_total_w",
            "power_idle_total_w",
            "FPGA TBP dyn.",
            C["platforms"]["fpga_dynamic"],
        ),
        "gpu": (
            "power_gpu_w",
            "idle_gpu_w",
            "GPU board dyn.",
            C["gpu_power_domains"]["gpu_board_dynamic"],
        ),
    }
    # Gather CPU RAPL dynamic separately (it accompanies GPU in deployment)
    rapl_dyn: Dict[str, float] = {}
    for _, row in df[df["platform"] == "gpu"].iterrows():
        sc = str(row["scenario"])
        rl = row.get("power_cpu_rapl_w")
        ri = row.get("idle_cpu_rapl_w")
        if rl and ri and not pd.isna(rl) and not pd.isna(ri):
            rapl_dyn[sc] = max(0.0, float(rl) - float(ri))

    scenarios = [
        s
        for s in SCENARIO_ORDER
        if any(not df[(df["platform"] == p) & (df["scenario"] == s)].empty for p in PLATFORM_SPECS)
    ]
    n = len(scenarios)
    x = np.arange(n)
    n_bars = len(PLATFORM_SPECS) + (1 if rapl_dyn else 0)
    w = 0.7 / n_bars

    fig, ax = plt.subplots(figsize=(11, 5))

    for i, (plat, (load_col, idle_col, label, color)) in enumerate(PLATFORM_SPECS.items()):
        vals = []
        for s in scenarios:
            row = df[(df["platform"] == plat) & (df["scenario"] == s)]
            if row.empty:
                vals.append(0.0)
            else:
                lw = row[load_col].values[0]
                iw = row[idle_col].values[0]
                vals.append(
                    max(0.0, float(lw) - float(iw))
                    if lw is not None and iw is not None and not pd.isna(lw) and not pd.isna(iw)
                    else 0.0
                )
        offset = (i - n_bars / 2 + 0.5) * w
        bars = ax.bar(x + offset, vals, w * 0.88, label=label, color=color, edgecolor="white")
        for bar, v in zip(bars, vals):
            if v > 0.05:
                ax.text(
                    bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.05,
                    f"{v:.2f}",
                    ha="center",
                    va="bottom",
                    fontsize=7.5,
                )

    if rapl_dyn:
        i = len(PLATFORM_SPECS)
        vals = [rapl_dyn.get(s, 0.0) for s in scenarios]
        offset = (i - n_bars / 2 + 0.5) * w
        bars = ax.bar(
            x + offset,
            vals,
            w * 0.88,
            label="CPU RAPL dyn. (GPU host)",
            color=C["gpu_power_domains"]["cpu_rapl_dynamic"],
            edgecolor="white",
            alpha=0.8,
        )
        for bar, v in zip(bars, vals):
            if v > 0.05:
                ax.text(
                    bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.05,
                    f"{v:.2f}",
                    ha="center",
                    va="bottom",
                    fontsize=7.5,
                )

    ax.set_xticks(x)
    ax.set_xticklabels(
        [SCENARIO_LABELS.get(s, s).replace("\n", " ") for s in scenarios], fontsize=9
    )
    ax.set_ylabel("Dynamic Power  (W)  [load − idle]")
    ax.set_title(
        f"Dynamic Power by Platform & Scenario — {MODEL_NAME}\n"
        "⚠  FPGA TBP and GPU board have different scopes (see §8 notes)"
    )
    ax.legend(fontsize=9)
    fig.tight_layout()
    if SAVE_FIGURES:
        plt.savefig(PLOTS_DIR / f"power_dynamic_comparison.{FIGURE_FORMAT}", bbox_inches="tight")
    plt.show()


plot_dynamic_power_comparison(df)


In [ ]:
# ── §8c  Power breakdown by scenario ─────────────────────────────────────────
# Grouped bar chart: GPU | CPU | FPGA side-by-side for each of the 5 scenarios.
#   GPU  : idle (pale blue) + dynamic (blue)
#   CPU  : idle (pale orange) + dynamic (orange)  [RAPL = full CPU power]
#   FPGA : idle base (pale green) + dynamic PL (dark) + dynamic PS (mid) + dynamic Peripherals (pale)
# Total power annotated on top of each bar.


def plot_power_breakdown_by_scenario(
    df: pd.DataFrame,
    scenario_order: List[str],
    scenario_labels: Dict[str, str],
    platform_labels: Dict[str, str],
    C: dict,
) -> Tuple[plt.Figure, plt.Axes]:
    # ── Colours ───────────────────────────────────────────────────────────────
    gpu_idle_c = C["platforms"]["gpu_idle"]
    gpu_dyn_c = C["platforms"]["gpu_dynamic"]
    cpu_idle_c = C["platforms"]["cpu_idle"]
    cpu_dyn_c = C["platforms"]["cpu_dynamic"]
    fpga_idle_c = C["platforms"]["fpga_idle"]
    fpga_PL_c = C["fpga_power_groups"]["PL"]
    fpga_PS_c = C["fpga_power_groups"]["PS"]
    fpga_peri_c = C["fpga_power_groups"]["peripherals"]

    n_sc = len(scenario_order)
    n_plt = 3  # GPU, CPU, FPGA
    bar_w = 0.22
    group_gap = 0.15  # extra space between scenario groups
    group_w = n_plt * bar_w + group_gap
    x_centers = np.arange(n_sc) * group_w
    offsets = np.array([-bar_w, 0, bar_w])  # GPU, CPU, FPGA

    fig, ax = plt.subplots(figsize=(12, 5))

    legend_handles = []
    legend_labels_list = []

    def _get(row, col, default=0.0):
        v = row[col] if row is not None and col in row.index else None
        return (
            float(v) if v is not None and not (isinstance(v, float) and np.isnan(v)) else default
        )

    for si, sc in enumerate(scenario_order):
        xc = x_centers[si]

        for pi, (plat, offset) in enumerate(zip(["gpu", "cpu", "fpga"], offsets)):
            mask = (df["platform"] == plat) & (df["scenario"] == sc)
            rows = df[mask]
            row = rows.iloc[0] if len(rows) > 0 else None

            xbar = xc + offset
            bottom = 0.0

            if plat == "gpu":
                idle = _get(row, "idle_gpu_w")
                total = _get(row, "power_gpu_w")
                dyn = max(total - idle, 0.0)
                b1 = ax.bar(
                    xbar,
                    idle,
                    bar_w,
                    bottom=bottom,
                    color=gpu_idle_c,
                    label="_gpu_idle" if si > 0 else "GPU idle",
                )
                bottom += idle
                b2 = ax.bar(
                    xbar,
                    dyn,
                    bar_w,
                    bottom=bottom,
                    color=gpu_dyn_c,
                    label="_gpu_dyn" if si > 0 else "GPU dynamic",
                )
                bottom += dyn
                if si == 0:
                    legend_handles += [b1, b2]
                    legend_labels_list += ["GPU idle", "GPU dynamic"]

            elif plat == "cpu":
                idle = _get(row, "idle_cpu_rapl_w")
                total = _get(row, "power_cpu_rapl_w")
                dyn = max(total - idle, 0.0)
                b1 = ax.bar(xbar, idle, bar_w, bottom=bottom, color=cpu_idle_c, label="_cpu_idle")
                bottom += idle
                b2 = ax.bar(xbar, dyn, bar_w, bottom=bottom, color=cpu_dyn_c, label="_cpu_dyn")
                bottom += dyn
                if si == 0:
                    legend_handles += [b1, b2]
                    legend_labels_list += ["CPU idle", "CPU dynamic"]

            else:  # fpga
                idle_pl = _get(row, "idle_PL_w")
                idle_ps = _get(row, "idle_PS_w")
                idle_peri = _get(row, "idle_peripherals_w")
                idle_base = idle_pl + idle_ps + idle_peri

                pl_total = _get(row, "power_PL_w")
                ps_total = _get(row, "power_PS_w")
                peri_total = _get(row, "power_peripherals_w")
                dyn_pl = max(pl_total - idle_pl, 0.0)
                dyn_ps = max(ps_total - idle_ps, 0.0)
                dyn_peri = max(peri_total - idle_peri, 0.0)

                b0 = ax.bar(
                    xbar, idle_base, bar_w, bottom=bottom, color=fpga_idle_c, label="_fpga_idle"
                )
                bottom += idle_base
                b1 = ax.bar(xbar, dyn_pl, bar_w, bottom=bottom, color=fpga_PL_c, label="_fpga_PL")
                bottom += dyn_pl
                b2 = ax.bar(xbar, dyn_ps, bar_w, bottom=bottom, color=fpga_PS_c, label="_fpga_PS")
                bottom += dyn_ps
                b3 = ax.bar(
                    xbar, dyn_peri, bar_w, bottom=bottom, color=fpga_peri_c, label="_fpga_peri"
                )
                bottom += dyn_peri
                if si == 0:
                    legend_handles += [b0, b1, b2, b3]
                    legend_labels_list += [
                        "FPGA idle",
                        "FPGA PL (dyn)",
                        "FPGA PS (dyn)",
                        "FPGA Peripherals (dyn)",
                    ]

            # ── Total annotation on top ───────────────────────────────────────
            total_val = _get(row, "power_total_w") if row is not None else bottom
            if total_val == 0.0:
                total_val = bottom
            ax.text(
                xbar,
                total_val + 0.3,
                f"{total_val:.1f}W",
                ha="center",
                va="bottom",
                fontsize=7,
                rotation=90,
            )

    # ── Axes cosmetics ────────────────────────────────────────────────────────
    ax.set_xticks(x_centers)
    ax.set_xticklabels([scenario_labels.get(s, s) for s in scenario_order])
    ax.set_ylabel("Power (W)")
    ax.set_title(f"Power breakdown by scenario — {MODEL_NAME}")
    ax.set_xlim(-group_w * 0.6, x_centers[-1] + group_w * 0.6)

    # Platform sub-labels between the xtick labels and the bars
    for si, sc in enumerate(scenario_order):
        xc = x_centers[si]
        for plat, offset in zip(["GPU", "CPU", "FPGA"], offsets):
            ax.text(
                xc + offset,
                -ax.get_ylim()[1] * 0.04,
                plat,
                ha="center",
                va="top",
                fontsize=6,
                color="grey",
                transform=ax.get_xaxis_transform(),
            )

    ax.legend(
        legend_handles, legend_labels_list, loc="upper right", fontsize=8, framealpha=0.8, ncol=2
    )
    fig.tight_layout()
    return fig, ax


fig_pw, ax_pw = plot_power_breakdown_by_scenario(
    df, SCENARIO_ORDER, SCENARIO_LABELS, PLATFORM_LABELS, C
)
if SAVE_FIGURES:
    PLOTS_DIR.mkdir(parents=True, exist_ok=True)
    fig_pw.savefig(PLOTS_DIR / f"power_breakdown_by_scenario.{FIGURE_FORMAT}", bbox_inches="tight")
    print(f"Saved power_breakdown_by_scenario.{FIGURE_FORMAT}")
plt.show()


## 9 · Energy per Inference

Energy per inference = $P_\text{avg} \times t_\text{tile}$ accounts for both power draw **and** latency — a platform that is fast but power-hungry can use less energy than a slow, low-power platform.

$$E_\text{tile}\;[\text{mJ}] = P_\text{avg}\;[\text{W}] \times t_\text{tile}\;[\text{ms}]$$

### Two complementary views

| View | Formula | What it answers |
|---|---|---|
| **Total energy** | $P_\text{load} \times t$ | Full deployment cost — what the system *actually* consumes per inference |
| **Dynamic energy** | $(P_\text{load} - P_\text{idle}) \times t$ | Incremental cost of one inference — relevant when the platform is always powered on |

**When does each matter?**
- **Satellite on-board**: FPGA is always powered during a pass (~10 min). Idle power accumulates between inferences → total energy and average power matter.
- **Data-centre GPU**: GPU is powered on 24/7. Marginal cost per additional job = dynamic energy.
- **Duty-cycle argument**: At low utilisation, total energy ≈ idle power × time. At high utilisation, total energy → dynamic × count. The **break-even duty cycle** where FPGA and GPU have equal total energy is the key operating point.

### ⚠ Scope warnings
1. **FPGA `TBP`** = MPSoC + MGT + Peripherals ≈ 11–19 W (whole ZCU102 board via INA226+PMBus).
2. **GPU `power_total`** = GPU board (nvidia-smi) + CPU RAPL (x86 package). Not additive with FPGA.
3. **CPU** = RAPL only (no GPU idle included). No platform has full wall-plug measurement.
4. **Do not rank FPGA vs GPU by absolute energy** without noting scope and duty-cycle context.

### §9a  Total energy (left) vs Dynamic energy (right)
Bars are stacked: **pale** = idle energy component ($P_\text{idle} \times t$),
**solid** = dynamic energy ($P_\text{dynamic} \times t$).
The pale layer shows how much of the bill goes to "keeping the lights on" — especially large for FPGA at slow inference rates.


In [ ]:
# ── §9a  Total energy vs Dynamic energy ───────────────────────────────────────
# Left panel:  total energy = P_load × t  (what the platform actually consumed)
# Right panel: dynamic energy = P_dynamic × t  (incremental inference cost)
#
# Colors from shared palette (Okabe-Ito, colorblind-safe):
#   FPGA TBP:  fpga_dynamic (solid) / fpga_idle (pale)
#   GPU total: gpu_dynamic  (solid) / gpu_idle  (pale)
#   CPU RAPL:  cpu_dynamic  (solid) / cpu_idle  (pale)
#
# For GPU: power_total_w = GPU board + CPU RAPL (both measured simultaneously);
#          power_idle_total_w = idle_gpu + idle_rapl.
# For CPU platform: power_total_w = RAPL only; power_idle_total_w = idle_rapl.
# For FPGA: power_total_w = TBP (INA226 + PMBus); power_idle_total_w = idle TBP.
#
# ⚠ Idle energy accumulates when the platform sits waiting between inferences.
#   At higher throughput (larger batches / continuous stream), idle energy
#   becomes negligible and only dynamic energy matters.

_ENERGY_PALETTE = {
    "fpga": {"load": C["platforms"]["fpga_dynamic"], "idle": C["platforms"]["fpga_idle"]},
    "gpu": {"load": C["platforms"]["gpu_dynamic"], "idle": C["platforms"]["gpu_idle"]},
    "cpu": {"load": C["platforms"]["cpu_dynamic"], "idle": C["platforms"]["cpu_idle"]},
}


def _energy_vals(
    df: pd.DataFrame, platform: str, scenario: str
) -> Tuple[Optional[float], Optional[float], Optional[float], Optional[float]]:
    """Return (total_mj, idle_mj, dynamic_mj, latency_ms) for one platform×scenario.

    Returns (None, None, None, None) if power data is missing.
    """
    row = df[(df["platform"] == platform) & (df["scenario"] == scenario)]
    if row.empty:
        return None, None, None, None
    r = row.iloc[0]
    lat = float(r["latency_ms"])
    p_load = r.get("power_total_w")
    p_idle = r.get("power_idle_total_w")
    if p_load is None or pd.isna(p_load):
        return None, None, None, lat
    p_load = float(p_load)
    total_mj = p_load * lat
    if p_idle is not None and not pd.isna(p_idle):
        p_idle = float(p_idle)
        idle_mj = p_idle * lat
        dyn_mj = max(0.0, (p_load - p_idle) * lat)
    else:
        idle_mj = None
        dyn_mj = None
    return total_mj, idle_mj, dyn_mj, lat


def plot_energy(df: pd.DataFrame, scenarios: Optional[List[str]] = None) -> None:
    """Side-by-side panels: total energy (left) and dynamic energy (right).

    Each bar = idle energy (pale) stacked with dynamic energy (solid).
    The pale portion represents energy spent keeping the platform at idle
    during the inference latency window.

    Interpretation:
      • A tall pale bar on FPGA = idle board power accumulated over slow inference.
      • A large solid bar on GPU = high incremental power × fast latency.
      • Equal bar heights mean equal energy per inference — but they have
        different measurement scopes (see §8 and §9 notes).
    """
    scenarios = scenarios or SCENARIO_ORDER
    platforms = ["fpga", "gpu", "cpu"]

    fig, (ax_total, ax_dyn) = plt.subplots(
        1,
        2,
        figsize=(16, 5.5),
        sharey=False,
    )

    n_sc = len(scenarios)
    n_pl = len(platforms)
    w = 0.7 / n_pl
    x = np.arange(n_sc)

    for panel_idx, (ax, use_dynamic) in enumerate([(ax_total, False), (ax_dyn, True)]):
        max_y = 0.0
        for i, plat in enumerate(platforms):
            palette = _ENERGY_PALETTE[plat]
            offset = (i - n_pl / 2 + 0.5) * w

            idle_vals = []
            dyn_vals = []
            total_vals = []

            for sc in scenarios:
                total_mj, idle_mj, dyn_mj, lat = _energy_vals(df, plat, sc)
                if total_mj is None:
                    idle_vals.append(0.0)
                    dyn_vals.append(0.0)
                    total_vals.append(0.0)
                    continue
                if idle_mj is not None:
                    idle_vals.append(idle_mj)
                    dyn_vals.append(dyn_mj or 0.0)
                else:
                    idle_vals.append(0.0)
                    dyn_vals.append(total_mj)
                total_vals.append(total_mj)

            idle_arr = np.array(idle_vals)
            dyn_arr = np.array(dyn_vals)

            if use_dynamic:
                # Right panel: dynamic energy only (no idle stacking)
                bars = ax.bar(
                    x + offset,
                    dyn_arr,
                    w * 0.88,
                    label=PLATFORM_LABELS.get(plat, plat),
                    color=palette["load"],
                    edgecolor="white",
                )
                for bar, v in zip(bars, dyn_arr):
                    if v > 50:
                        ax.text(
                            bar.get_x() + bar.get_width() / 2,
                            bar.get_height() + 20,
                            f"{v:.0f}",
                            ha="center",
                            va="bottom",
                            fontsize=7.5,
                            color=palette["load"],
                        )
                max_y = max(max_y, float(np.max(dyn_arr)))
            else:
                # Left panel: stacked idle (pale) + dynamic (solid)
                ax.bar(
                    x + offset,
                    idle_arr,
                    w * 0.88,
                    label=f"{PLATFORM_LABELS.get(plat, plat)} (idle)",
                    color=palette["idle"],
                    edgecolor="white",
                )
                bars = ax.bar(
                    x + offset,
                    dyn_arr,
                    w * 0.88,
                    bottom=idle_arr,
                    label=f"{PLATFORM_LABELS.get(plat, plat)} (dynamic)",
                    color=palette["load"],
                    edgecolor="white",
                )
                for bar, v_idle, v_dyn in zip(bars, idle_arr, dyn_arr):
                    total = v_idle + v_dyn
                    if total > 50:
                        ax.text(
                            bar.get_x() + bar.get_width() / 2,
                            total + 20,
                            f"{total:.0f}",
                            ha="center",
                            va="bottom",
                            fontsize=7.5,
                            color=palette["load"],
                        )
                max_y = max(max_y, float(np.max(idle_arr + dyn_arr)))

        ax.set_xticks(x)
        ax.set_xticklabels(
            [SCENARIO_LABELS.get(s, s).replace("\n", " ") for s in scenarios], fontsize=9
        )
        ax.set_ylabel("Energy per inference  (mJ)")
        ax.set_ylim(bottom=0, top=max_y * 1.18)

        if use_dynamic:
            ax.set_title(
                "Dynamic energy  =  $(P_{load} - P_{idle}) \\times t$\n"
                "Incremental cost of one inference (static overhead removed)"
            )
        else:
            ax.set_title(
                "Total energy  =  $P_{load} \\times t$\n"
                "Pale = idle component  ·  Solid = dynamic component"
            )

        # Deduplicate legend
        handles, labels_l = ax.get_legend_handles_labels()
        by_label = dict(zip(labels_l, handles))
        ax.legend(by_label.values(), by_label.keys(), fontsize=7.5, ncol=1, loc="upper right")

    fig.suptitle(
        f"Energy per Inference — {MODEL_NAME}\n"
        "⚠  FPGA scope = TBP (full board)  ·  GPU scope = GPU board + CPU RAPL  ·  "
        "CPU scope = RAPL only",
        fontsize=10,
        y=1.02,
    )
    fig.tight_layout()
    if SAVE_FIGURES:
        plt.savefig(PLOTS_DIR / f"energy_per_tile.{FIGURE_FORMAT}", bbox_inches="tight")
    plt.show()

    # ── Key insight printout ───────────────────────────────────────────────────
    print("── Energy per inference summary ──────────────────────────────────────")
    print(
        f"{'Platform':8s}  {'Scenario':15s}  {'Total(mJ)':>10s}  {'Dynamic(mJ)':>12s}  "
        f"{'Idle%':>7s}  {'Latency(ms)':>12s}"
    )
    print("─" * 75)
    for sc in scenarios:
        for plat in platforms:
            total_mj, idle_mj, dyn_mj, lat = _energy_vals(df, plat, sc)
            if total_mj is None or lat is None:
                continue
            idle_frac = (idle_mj / total_mj * 100) if (idle_mj and total_mj) else 0.0
            dyn_str = f"{dyn_mj:.0f}" if dyn_mj is not None else "  N/A"
            print(
                f"{plat:8s}  {sc:15s}  {total_mj:>10.0f}  {dyn_str:>12s}  "
                f"{idle_frac:>6.1f}%  {lat:>12.1f}"
            )
        print()


plot_energy(df)


## 9b · Power Analysis — Reflection & What's Still Missing

### Key insights from §8–§9

| Observation | Evidence |
|---|---|
| **PL fabric is the only dynamic FPGA component** | PL varies 9.5–13.8 W across scenarios; PS, MGT, Peripherals barely change (< 0.3 W swing) |
| **PS_compute increment confirms ARM A53 entropy load** | PS_compute dynamic ≈ +0.12–0.17 W during entropy-heavy scenarios (within noise, but consistent direction) |
| **FPGA and GPU have comparable dynamic energy for the full scenario** | FPGA: ~1.6 J, GPU: ~1.1 J (full scenario) — within 50%, despite 10× latency difference |
| **GPU NN is ~2× more dynamic-energy-efficient than FPGA for pure NN** | `nn_only` dynamic: FPGA 605 mJ vs GPU 508 mJ — GPU wins slightly even at batch=1 |
| **CPU is 6–20× worse than GPU/FPGA in dynamic energy** | CPU full dynamic: 10 110 mJ vs GPU 1 115 mJ |
| **FPGA idle fraction is large (60–77%)** | Running at < 100% duty cycle costs disproportionate energy; FPGA is only efficient in continuous-throughput mode |

### ⚠ What is NOT covered — limitations to be aware of

**1. Duty-cycle / break-even analysis (not yet plotted)**
The current plots show single-inference energy. The *total system energy* over a mission depends on duty cycle $\delta \in [0,1]$:
$$P_\text{system} = P_\text{idle} + \delta \cdot P_\text{dynamic}$$
At low duty cycle, FPGA is penalised by large idle power (~11 W). A break-even plot ($\delta$ vs total energy FPGA vs GPU) would show the crossover point. **Not yet implemented.**

**2. Energy efficiency in ops/J (TOPS/W)**
We have `total_workload_ops` from the FPGA JSON (xmodel metadata). Dividing by dynamic energy gives a TOPS/W metric for the DPU. This would be the fairest comparison to published FPGA accelerator benchmarks. **Not yet implemented.**

**3. GPU batch-size scaling**
At batch=1, the GPU is severely under-utilised. GPU dynamic power would remain roughly constant for small batches but latency → latency/N → dynamic energy/tile would drop sharply. A batch-scaling curve would contextualise the GPU numbers. **Not measured** (DPU only supports batch=1, so comparison would be one-sided).

**4. Temperature and long-run stability**
All measurements are at steady-state during a 34–34s workload window. No thermal throttling was observed, but a long continuous-run (e.g., 10 min sustained) may show PL power creep. **Not measured.**

**5. Full system wall-plug power**
All platforms lack a common external measurement. An AC watt-meter on the FPGA barrel jack and on the host PSU would give true system power including VRM losses (~10–15%), fans (~5 W), SSD, motherboard. This is needed for true apples-to-apples. **Not measured — requires hardware instrumentation.**

**6. Energy per unit quality (J/dB)**
The model runs at INT8 quantised precision on FPGA vs FP32 on GPU/CPU. Energy per inference at equal reconstruction quality would require the RD-curve and quality metrics to be coupled with energy. Covered in `notebooks/RD-curve_ablation.ipynb` but not yet integrated here.


In [ ]:
def build_summary_table(df: pd.DataFrame, scenario: str = "full") -> pd.DataFrame:
    """Build a publication-ready summary table for one scenario."""
    subset = df[df["scenario"] == scenario].sort_values("platform")

    rows = []
    for _, row in subset.iterrows():
        plat = str(row["platform"])

        # Dynamic power
        p_dyn = None
        e_dyn = None
        idle_w = row.get("power_idle_total_w")
        total_w = row.get("power_total_w")
        if (
            total_w is not None
            and not pd.isna(total_w)
            and idle_w is not None
            and not pd.isna(idle_w)
        ):
            p_dyn = total_w - idle_w
            e_dyn = p_dyn * (row["latency_ms"] / 1000) * 1000  # mJ

        rows.append(
            {
                "Platform": PLATFORM_LABELS.get(plat, plat),
                "Precision": "INT8" if plat == "fpga" else "FP32",
                "Total Latency (ms)": round(row["latency_ms"], 2),
                "NN Latency (ms)": round(row["nn_latency_ms"], 2),
                "CPU Latency (ms)": round(row["cpu_latency_ms"], 2),
                "NN Fraction (%)": round(row["nn_fraction"] * 100, 1),
                "Throughput (p/s)": round(row["throughput_fps"], 2),
                "BPP": round(row["bpp"], 4) if row["bpp"] else None,
                "Total Power (W)": round(total_w, 1) if total_w and not pd.isna(total_w) else None,
                "Idle Power (W)": round(idle_w, 1) if idle_w and not pd.isna(idle_w) else None,
                "Energy/tile (mJ)": round(row["energy_per_tile_mj"], 0)
                if row["energy_per_tile_mj"] and not pd.isna(row["energy_per_tile_mj"])
                else None,
                "Dyn. Energy/tile (mJ)": round(e_dyn, 0) if e_dyn else None,
            }
        )

    return pd.DataFrame(rows).set_index("Platform")


# ── Full scenario summary ──────────────────────────────────────────────────────
print("═" * 80)
print(f"  SUMMARY — Full Scenario — {MODEL_NAME}")
print("═" * 80)
summary_full = build_summary_table(df, "full")
display(summary_full)

# ── Operational scenario: FPGA compress + GPU/CPU decompress ───────────────────
print("\n" + "═" * 80)
print(f"  OPERATIONAL COMPARISON — {MODEL_NAME}")
print("  Satellite: FPGA compresses (uplink) → Ground: GPU/CPU decompresses")
print("═" * 80)

operational_rows = []

# FPGA compress
fpga_comp = df[(df["platform"] == "fpga") & (df["scenario"] == "compress")]
if not fpga_comp.empty:
    r = fpga_comp.iloc[0]
    operational_rows.append(
        {
            "Role": "Satellite (compress)",
            "Platform": "FPGA (ZCU102)",
            "Latency (ms)": round(r["latency_ms"], 2),
            "Throughput (p/s)": round(r["throughput_fps"], 2),
            "Power (W)": round(r["power_total_w"], 1)
            if r["power_total_w"] and not pd.isna(r["power_total_w"])
            else None,
            "Energy/tile (mJ)": round(r["energy_per_tile_mj"], 0)
            if r["energy_per_tile_mj"] and not pd.isna(r["energy_per_tile_mj"])
            else None,
        }
    )

# GPU decompress
gpu_decomp = df[(df["platform"] == "gpu") & (df["scenario"] == "decompress")]
if not gpu_decomp.empty:
    r = gpu_decomp.iloc[0]
    operational_rows.append(
        {
            "Role": "Ground (decompress)",
            "Platform": "GPU (RTX A4000)",
            "Latency (ms)": round(r["latency_ms"], 2),
            "Throughput (p/s)": round(r["throughput_fps"], 2),
            "Power (W)": round(r["power_total_w"], 1)
            if r["power_total_w"] and not pd.isna(r["power_total_w"])
            else None,
            "Energy/tile (mJ)": round(r["energy_per_tile_mj"], 0)
            if r["energy_per_tile_mj"] and not pd.isna(r["energy_per_tile_mj"])
            else None,
        }
    )

# CPU decompress
cpu_decomp = df[(df["platform"] == "cpu") & (df["scenario"] == "decompress")]
if not cpu_decomp.empty:
    r = cpu_decomp.iloc[0]
    operational_rows.append(
        {
            "Role": "Ground (decompress)",
            "Platform": "CPU (x86 host)",
            "Latency (ms)": round(r["latency_ms"], 2),
            "Throughput (p/s)": round(r["throughput_fps"], 2),
            "Power (W)": round(r["power_total_w"], 1)
            if r["power_total_w"] and not pd.isna(r["power_total_w"])
            else None,
            "Energy/tile (mJ)": round(r["energy_per_tile_mj"], 0)
            if r["energy_per_tile_mj"] and not pd.isna(r["energy_per_tile_mj"])
            else None,
        }
    )

if operational_rows:
    op_df = pd.DataFrame(operational_rows).set_index("Role")
    display(op_df)

In [ ]:
# ── Export summaries as CSV ─────────────────────────────────────────────────────
if SAVE_FIGURES:
    PLOTS_DIR.mkdir(parents=True, exist_ok=True)
    summary_full.to_csv(PLOTS_DIR / "summary_full.csv")
    if operational_rows:
        op_df.to_csv(PLOTS_DIR / "summary_operational.csv")
    print(f"Saved CSVs to {PLOTS_DIR}/")
else:
    print(f"Set SAVE_FIGURES = True to export CSVs and {FIGURE_FORMAT.upper()} figures.")

---
## 11 · FPGA Parallel vs Sequential Comparison

> **Standalone section — independent of the main `df` DataFrame.**
> Sequential result files (`benchmark_fpga_<scenario>_sequential.json`) are loaded
> directly from disk here and are **excluded from all §2–§10 analyses** (overview
> table, latency plots, power charts, summary tables).  Only this section reads them.

All FPGA scenarios use **parallel real‖imag execution by default** via Python threads.
`execute_async` releases the GIL while waiting on hardware, so two threads calling
`runner.run()` simultaneously genuinely overlap on the ZCU102's three DPU cores.

### Confirmed speedup (both g_a and g_s — updated results)

| Stage | Sequential | Parallel (default) | Speedup |
|---|---|---|---|
| `dpu_g_a` | ~74.0 ms | ~39.1 ms | **~1.9×** |
| `dpu_g_s` | ~73.6 ms | ~38.9 ms | **~1.9×** |
| Total DPU | ~151 ms | ~82 ms | **~1.8×** |
| Total | ~406 ms | ~338 ms | **~1.2×** |

> **Bug found & fixed:** `g_s` was not benefiting from parallelism because VART assigns DPU
> cores via **static round-robin** at `Runner.create_runner()` call time.  The old creation
> order `(g_a_1, g_s_1)` caused `g_s_1` to land on the **same** core as `g_s`, giving no
> overlap.  Fix: `for key in ("g_s", "g_a"):` — `g_s_1` is created first, landing on a
> core ≠ g_s (core 1 vs core 2).

### Generating the sequential baseline

`run_full_benchmark.py` now runs **both parallel and sequential** modes by default.  To
skip one:
```bash
python run_full_benchmark.py --no-fpga-sequential   # parallel only (fast)
python run_full_benchmark.py --no-fpga-parallel     # sequential only
```
Sequential files are saved as `benchmark_fpga_<scenario>_sequential.json`.

### Sub-sections

| Cell | Content |
|---|---|
| **§11a** | `full` — complete pipeline, parallel vs sequential |
| **§11b** | `compress` — encode path only |
| **§11d** | 4-way comparison: GPU / CPU / FPGA-parallel / FPGA-sequential |


In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# §11 helper: parallel ‖ vs sequential comparison for any FPGA scenario
# ──────────────────────────────────────────────────────────────────────────────

# Fraction of figure width reserved for the legend column on the right.
# Increase to give more room to the legend; decrease to widen the bars.
_LEGEND_WIDTH_FRAC = 0.1


def _load_fpga_scenario(scenario: str) -> Tuple[Optional[dict], Optional[dict]]:
    """Return (parallel_dict, sequential_dict | None) for an FPGA scenario.

    Each dict has keys: label, latency_ms, nn_latency_ms, cpu_latency_ms, steps.
    """
    par_row = df[(df["platform"] == "fpga") & (df["scenario"] == scenario)]

    seq_path = BENCHMARK_DIR / f"benchmark_fpga_{scenario}_sequential.json"
    seq_raw: Optional[dict] = None
    if seq_path.exists():
        with open(seq_path) as _f:
            seq_raw = json.load(_f)

    par_data: Optional[dict] = None
    if not par_row.empty:
        r = par_row.iloc[0]
        par_data = {
            "label": "parallel",
            "latency_ms": float(r["latency_ms"]),
            "nn_latency_ms": float(r["nn_latency_ms"]),
            "cpu_latency_ms": float(r["cpu_latency_ms"]),
            "steps": r["_step_breakdown"],
        }

    seq_data: Optional[dict] = None
    if seq_raw is not None:
        seq_data = {
            "label": "sequential",
            "latency_ms": float(seq_raw.get("latency_total_mean_ms", 0.0)),
            "nn_latency_ms": float(seq_raw.get("latency_dpu_total_mean_ms", 0.0)),
            "cpu_latency_ms": float(seq_raw.get("latency_cpu_total_mean_ms", 0.0)),
            "steps": _extract_step_breakdown(seq_raw, "fpga"),
        }

    return par_data, seq_data


def compare_scenario(scenario: str) -> None:
    """Print summary table and step-breakdown chart for a parallel vs sequential FPGA run."""
    par, seq = _load_fpga_scenario(scenario)

    variants = [v for v in (par, seq) if v is not None]
    if not variants:
        print(f"⚠  No data for scenario '{scenario}'.  Run the benchmark first.")
        return

    # ── Summary table ─────────────────────────────────────────────────────────
    print(f"\n  Scenario: {scenario}")
    print(f"  {'Mode':<40s}  {'Total (ms)':>10s}  {'DPU (ms)':>9s}  {'CPU (ms)':>9s}")
    print("  " + "─" * 74)
    for v in variants:
        print(
            f"  {v['label']:<40s}  {v['latency_ms']:10.2f}  "
            f"{v['nn_latency_ms']:9.2f}  {v['cpu_latency_ms']:9.2f}"
        )

    if par is not None and seq is not None:
        print()
        for key in ("nn_g_a", "nn_g_s", "nn_h_a", "nn_h_s"):
            v_par = par["steps"].get(key, 0.0)
            v_seq = seq["steps"].get(key, 0.0)
            if v_par > 0.5 or v_seq > 0.5:
                gain = (v_seq - v_par) / v_seq * 100 if v_seq > 0 else 0.0
                print(
                    f"    {key:10s}:  seq={v_seq:.1f} ms  par={v_par:.1f} ms  "
                    f"saved={v_seq - v_par:.1f} ms ({gain:.0f}%)"
                )
        saved = seq["latency_ms"] - par["latency_ms"]
        pct = saved / seq["latency_ms"] * 100 if seq["latency_ms"] > 0 else 0.0
        print(
            f"    {'Total':10s}:  seq={seq['latency_ms']:.1f} ms  "
            f"par={par['latency_ms']:.1f} ms  saved={saved:.1f} ms ({pct:.0f}%)"
        )

    # ── Step breakdown chart ───────────────────────────────────────────────────
    n = len(variants)
    fig, ax = plt.subplots(figsize=(12, max(2, n * 1.2)))

    max_lat = max(v["latency_ms"] for v in variants)
    y_labels = []

    for idx, v in enumerate(variants):
        steps = v["steps"]
        ordered = [(s, steps.get(s, 0.0)) for s in STEP_ORDER if steps.get(s, 0.0) >= 0.1]
        known = {s for s, _ in ordered}
        for s, val in sorted(steps.items()):
            if s not in known and val >= 0.1:
                ordered.append((s, val))

        left = 0.0
        for step_name, ms_val in ordered:
            color = STEP_COLORS.get(step_name, "#bdc3c7")
            ax.barh(
                idx,
                ms_val,
                left=left,
                height=0.6,
                color=color,
                edgecolor="white",
                linewidth=0.5,
                label=step_name,
            )
            if ms_val > max_lat * 0.05:
                ax.text(
                    left + ms_val / 2,
                    idx,
                    f"{ms_val:.1f}",
                    ha="center",
                    va="center",
                    fontsize=7.5,
                    color="white",
                    fontweight="bold",
                )
            left += ms_val

        y_labels.append(f"{v['label']}\n({v['latency_ms']:.1f} ms)")

    ax.set_yticks(range(n))
    ax.set_yticklabels(y_labels, fontsize=9)
    ax.set_xlabel("Latency (ms)")
    ax.set_xlim(0, max_lat * 1.02)
    ax.invert_yaxis()
    ax.set_title(f"FPGA parallel ‖ vs sequential — {scenario} — {MODEL_NAME}", fontsize=11)

    # Legend lives outside the axes in the right margin.
    # _LEGEND_WIDTH_FRAC controls how much space to reserve (tweak at the top).
    handles, lbls = ax.get_legend_handles_labels()
    by_label = dict(zip(lbls, handles))
    fig.legend(
        by_label.values(),
        by_label.keys(),
        loc="center left",
        bbox_to_anchor=(1.0 - _LEGEND_WIDTH_FRAC, 0.5),
        fontsize=7,
        ncol=1,
        frameon=True,
    )
    # Shrink the axes so it doesn't overlap the legend column
    fig.subplots_adjust(right=1.0 - _LEGEND_WIDTH_FRAC - 0.01)

    if SAVE_FIGURES:
        PLOTS_DIR.mkdir(parents=True, exist_ok=True)
        plt.savefig(
            PLOTS_DIR / f"fpga_parallel_vs_sequential_{scenario}.{FIGURE_FORMAT}",
            bbox_inches="tight",
        )
    plt.show()


# ── §11a: full pipeline ────────────────────────────────────────────────────────
compare_scenario("full")


In [ ]:
compare_scenario("nn_only")
compare_scenario("compress")
compare_scenario("decompress")

In [ ]:
# ── §11d: 4-way latency comparison ────────────────────────────────────────────
# GPU / CPU (ZCU102 ARM) / FPGA-parallel / FPGA-sequential for the three
# cross-platform scenarios: full pipeline, compress-only, decompress-only.

_COMPARE_SC = ["full", "compress", "decompress", "nn_only"]
_COMPARE_LBL = {
    "full": "Full",
    "compress": "Compress",
    "decompress": "Decompress",
    "nn_only": "DPU only",
}

lat_4way: Dict[str, Dict[str, float]] = {sc: {} for sc in _COMPARE_SC}
for sc in _COMPARE_SC:
    for plat in ("gpu", "cpu", "fpga"):
        row = df[(df["platform"] == plat) & (df["scenario"] == sc)]
        if not row.empty:
            lat_4way[sc][plat] = float(row.iloc[0]["latency_ms"])

    seq_path = BENCHMARK_DIR / f"benchmark_fpga_{sc}_sequential.json"
    if seq_path.exists():
        with open(seq_path) as _f:
            d = json.load(_f)
        lat_4way[sc]["fpga_seq"] = float(d.get("latency_total_mean_ms", 0.0))

platform_info_4way: List[Tuple[str, str, str]] = [
    ("gpu", "GPU", "#2196F3"),
    ("cpu", "CPU", "#FF9800"),
    ("fpga", "FPGA (parallel)", "#4CAF50"),
    ("fpga_seq", "FPGA (sequential)", "#A5D6A7"),
]

n_sc = len(_COMPARE_SC)
n_plat = len(platform_info_4way)
x = np.arange(n_sc)
width = 0.18

fig, ax = plt.subplots(figsize=(10, 5))
for i, (plat, lbl, clr) in enumerate(platform_info_4way):
    vals = [lat_4way[sc].get(plat, float("nan")) for sc in _COMPARE_SC]
    bars = ax.bar(
        x + (i - n_plat / 2 + 0.5) * width, vals, width=width * 0.9, color=clr, label=lbl, zorder=3
    )
    for bar, v in zip(bars, vals):
        if not np.isnan(v):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 3,
                f"{v:.0f}",
                ha="center",
                va="bottom",
                fontsize=7.5,
                fontweight="bold",
            )

ax.set_xticks(x)
ax.set_xticklabels([_COMPARE_LBL[sc] for sc in _COMPARE_SC], fontsize=10)
ax.set_ylabel("Total latency (ms)")
ax.set_title(f"GPU / CPU / FPGA-parallel / FPGA-sequential — {MODEL_NAME}", fontsize=12)
ax.legend(loc="upper right", fontsize=9)
ax.yaxis.grid(True, alpha=0.35, zorder=0)
ax.set_axisbelow(True)
fig.tight_layout()
if SAVE_FIGURES:
    PLOTS_DIR.mkdir(parents=True, exist_ok=True)
    plt.savefig(PLOTS_DIR / f"fpga_4way_latency_comparison.{FIGURE_FORMAT}", bbox_inches="tight")
plt.show()